# 📘 CI/CD & Version Control for Data Pipelines
## Databricks Notebook — Production-Grade Pipeline Deployment

**What you'll learn:**
- Git repository structure for data engineering projects
- Code organization patterns (modular pipelines)
- Environment management (dev/staging/prod)
- Testing strategies for data pipelines
- CI/CD pipeline design (conceptual + implementable patterns)
- Database migrations as code
- Feature flags for pipelines
- Databricks-specific deployment patterns (Declarative Automation Bundles)

**Note:** Git commands and CI/CD tools can't run inside Databricks Community Edition.
We focus on code patterns, structures, and designs that ARE implementable.

**Prerequisites:** Notebooks 01-27 (techmart_dw database)

---

---
# 📗 Section 1: Why CI/CD for Data Engineering

## The Problem Without CI/CD

| Scenario | Without CI/CD | With CI/CD |
|---|---|---|
| Code change | Push directly to prod, hope it works | Automated tests catch issues before deploy |
| Multiple devs | Overwrite each other's work | Git branching + merge reviews |
| Rollback | Manual, error-prone, hours of downtime | One-click rollback in minutes |
| Environments | "Works on my machine" | Identical dev/staging/prod configs |
| Data quality | Find issues days later from angry users | Automated quality gates block bad deploys |

## CI/CD for Data vs. Software Engineering

Data pipelines have unique challenges:
- **Data dependencies** — upstream schema changes break downstream
- **Stateful processing** — can't just redeploy, need to handle existing data
- **Volume testing** — unit tests pass but pipeline fails at scale
- **Environment parity** — prod has 100M rows, dev has 1000
- **Schema evolution** — tables change over time, migrations needed

In [0]:
# The cost of NOT having CI/CD — a real scenario simulation
import json
from datetime import datetime

# Simulate a deployment log without CI/CD
deployment_incidents = [
    {
        "date": "2024-01-15",
        "change": "Modified customer_id join logic",
        "result": "FAILURE",
        "impact": "3 downstream dashboards broken for 2 days",
        "root_cause": "No tests caught the NULL handling change",
        "time_to_fix": "16 hours",
        "cost_estimate": "$45,000 (lost reporting + engineer time)"
    },
    {
        "date": "2024-02-20",
        "change": "Added new column to orders table",
        "result": "FAILURE",
        "impact": "ETL pipeline crashed, 500K records lost",
        "root_cause": "Schema change not propagated to downstream",
        "time_to_fix": "8 hours",
        "cost_estimate": "$25,000 (data recovery + downtime)"
    },
    {
        "date": "2024-03-10",
        "change": "Updated aggregation logic",
        "result": "PARTIAL FAILURE",
        "impact": "Revenue numbers off by 12% for a week",
        "root_cause": "No integration test with real data volumes",
        "time_to_fix": "3 days",
        "cost_estimate": "$80,000 (wrong business decisions made)"
    }
]

print("=" * 70)
print("DEPLOYMENT INCIDENT LOG (Without CI/CD)")
print("=" * 70)
total_cost = 0
for incident in deployment_incidents:
    print(f"
📅 {incident['date']} — {incident['change']}")
    print(f"   Result: ❌ {incident['result']}")
    print(f"   Impact: {incident['impact']}")
    print(f"   Root Cause: {incident['root_cause']}")
    print(f"   Time to Fix: {incident['time_to_fix']}")
    print(f"   Cost: {incident['cost_estimate']}")
    cost = int(incident['cost_estimate'].split('$')[1].split(' ')[0].replace(',', ''))
    total_cost += cost

print(f"
{'=' * 70}")
print(f"TOTAL COST OF NO CI/CD (Q1 2024): ${total_cost:,}")
print(f"{'=' * 70}")
print("
✅ With CI/CD: All 3 incidents would have been caught before production.")


## CI/CD for Data Engineering -- The Full Picture

### What is CI/CD?

**Continuous Integration (CI):** Automatically test every code change
**Continuous Delivery (CD):** Automatically deploy tested code to environments

```
Developer pushes code
        |
        v
CI Pipeline runs:
  1. Lint (code style check)
  2. Unit tests (test individual functions)
  3. Integration tests (test with real data)
  4. Bundle validation (check DABs config)
        |
        v (if all pass)
CD Pipeline runs:
  1. Deploy to DEV (automatic)
  2. Run smoke tests on DEV
  3. Deploy to STAGING (automatic)
  4. Run regression tests on STAGING
  5. Deploy to PROD (manual approval)
```

### Why CI/CD Matters for Data Engineering

| Without CI/CD | With CI/CD |
|--------------|------------|
| "Works on my machine" | Tested in clean environment |
| Manual deployments (error-prone) | Automated, consistent deployments |
| No rollback plan | Easy rollback to previous version |
| "Who broke the pipeline?" | Every change tracked, tested |
| Deployment takes hours | Deployment takes minutes |
| Fear of deploying on Fridays | Deploy any time with confidence |

### The Data Engineering CI/CD Stack

```
Code: Python, SQL, YAML
Version Control: Git + GitHub/GitLab/Bitbucket
CI/CD: GitHub Actions / Azure DevOps / Jenkins
Testing: pytest + Great Expectations
Deployment: Databricks Asset Bundles (DABs)
Monitoring: Databricks system tables + Datadog
```

---
# 📗 Section 2: Git for Data Engineering

## Repository Structure

A well-organized repo is the foundation of good CI/CD.

```
techmart-data-platform/
├── README.md
├── pyproject.toml              # Project metadata & dependencies
├── Makefile                    # Common commands (test, lint, deploy)
├── .gitignore                  # What NOT to track
├── .github/
│   └── workflows/
│       ├── ci.yml              # Continuous Integration
│       └── cd.yml              # Continuous Deployment
├── src/
│   ├── __init__.py
│   ├── pipelines/
│   │   ├── __init__.py
│   │   ├── bronze/             # Raw ingestion
│   │   │   ├── ingest_orders.py
│   │   │   └── ingest_customers.py
│   │   ├── silver/             # Cleaned/conformed
│   │   │   ├── clean_orders.py
│   │   │   └── clean_customers.py
│   │   └── gold/               # Business aggregates
│   │       ├── daily_sales.py
│   │       └── customer_360.py
│   ├── transformations/        # Reusable transform functions
│   │   ├── __init__.py
│   │   ├── cleaning.py
│   │   └── aggregations.py
│   ├── quality/                # Data quality checks
│   │   ├── __init__.py
│   │   └── expectations.py
│   └── utils/                  # Shared utilities
│       ├── __init__.py
│       ├── config.py
│       └── logging.py
├── tests/
│   ├── unit/
│   │   ├── test_cleaning.py
│   │   └── test_aggregations.py
│   ├── integration/
│   │   └── test_pipeline_e2e.py
│   └── fixtures/
│       └── sample_data.json
├── config/
│   ├── dev.yaml
│   ├── staging.yaml
│   └── prod.yaml
├── migrations/
│   ├── 001_create_bronze_tables.sql
│   ├── 002_add_customer_segment.sql
│   └── 003_create_gold_views.sql
└── docs/
    ├── architecture.md
    └── runbook.md
```

In [0]:
# Implement the repository structure as a Python dict (design exercise)
repo_structure = {
    "techmart-data-platform": {
        "src": {
            "pipelines": {
                "bronze": ["ingest_orders.py", "ingest_customers.py", "ingest_products.py"],
                "silver": ["clean_orders.py", "clean_customers.py", "conform_products.py"],
                "gold": ["daily_sales.py", "customer_360.py", "product_performance.py"]
            },
            "transformations": ["cleaning.py", "aggregations.py", "enrichment.py"],
            "quality": ["expectations.py", "schema_validator.py"],
            "utils": ["config.py", "logging_utils.py", "spark_helpers.py"]
        },
        "tests": {
            "unit": ["test_cleaning.py", "test_aggregations.py", "test_enrichment.py"],
            "integration": ["test_orders_pipeline.py", "test_customer_pipeline.py"],
            "fixtures": ["sample_orders.json", "sample_customers.json"]
        },
        "config": ["dev.yaml", "staging.yaml", "prod.yaml"],
        "migrations": ["001_create_bronze.sql", "002_create_silver.sql", "003_create_gold.sql"],
        "docs": ["architecture.md", "runbook.md", "onboarding.md"]
    }
}

def print_tree(structure, prefix="", is_last=True):
    """Print a directory tree from a nested dict."""
    if isinstance(structure, dict):
        items = list(structure.items())
        for i, (name, content) in enumerate(items):
            connector = "└── " if i == len(items) - 1 else "├── "
            print(f"{prefix}{connector}{name}/")
            extension = "    " if i == len(items) - 1 else "│   "
            print_tree(content, prefix + extension, i == len(items) - 1)
    elif isinstance(structure, list):
        for i, item in enumerate(structure):
            connector = "└── " if i == len(structure) - 1 else "├── "
            print(f"{prefix}{connector}{item}")

print("📁 TechMart Data Platform Repository Structure:")
print_tree(repo_structure)


## Branching Strategies

### Trunk-Based Development (Recommended for DE)
```
main ─────●─────●─────●─────●─────●─────→
           \   /       \   /       \   /
feature/    ●─●     fix/ ●     feature/ ●─●
add-col              nulls       new-agg
```
- Short-lived branches (1-3 days max)
- Frequent merges to main
- Feature flags for incomplete work
- Best for: small teams, fast iteration

### GitFlow (For Regulated Environments)
```
main    ─────────────●───────────────●────→
                    / \             / \
release ──────────●───●───────────●───●──→
                 /                /
develop ──●──●──●──●──●──●──●──●──●──●──→
          \  /     \  /     \  /
feature    ●●       ●●       ●●
```
- Long-lived develop branch
- Release branches for stabilization
- Best for: large teams, compliance requirements

In [0]:
# .gitignore for Data Engineering projects
gitignore_content = """
# ===== DATA ENGINEERING .gitignore =====

# --- Python ---
__pycache__/
*.py[cod]
*$py.class
*.egg-info/
dist/
build/
.eggs/
venv/
.venv/
env/

# --- IDE ---
.vscode/
.idea/
*.swp
*.swo

# --- Data (NEVER commit data files) ---
*.csv
*.parquet
*.json.gz
*.avro
data/raw/
data/processed/
data/output/

# --- Credentials (CRITICAL — never commit) ---
.env
.env.*
*.pem
*.key
credentials.json
service_account.json
secrets/

# --- Databricks ---
.databricks/
*.dbc

# --- Logs ---
logs/
*.log

# --- OS ---
.DS_Store
Thumbs.db

# --- Large files ---
*.zip
*.tar.gz
*.7z

# --- Jupyter ---
.ipynb_checkpoints/
*.ipynb  # Version notebooks as .py files instead
"""

print("📄 .gitignore for Data Engineering:")
print(gitignore_content)
print("\n✅ Key rules:")
print("  1. NEVER commit data files (use cloud storage)")
print("  2. NEVER commit credentials (use secrets manager)")
print("  3. NEVER commit large files (use Git LFS if needed)")
print("  4. DO commit: code, configs, schemas, tests, docs")


In [0]:
# ============================================
# 🎯 YOUR TURN: Design a Repository Structure
# ============================================
# Design a Git repository structure for a new project:
# - Company: FinanceCorps (financial data platform)
# - Pipelines: transaction ingestion, fraud detection, regulatory reporting
# - Environments: dev, staging, prod
# - Requirements: must include tests, configs, migrations, docs
#
# Create a nested dict representing the repo structure and print it as a tree.
#
# Write your code below:


In [0]:
# ============================================
# ✅ SOLUTION
# ============================================
finance_repo = {
    "financecorps-data-platform": {
        "src": {
            "pipelines": {
                "ingestion": ["ingest_transactions.py", "ingest_accounts.py", "ingest_market_data.py"],
                "fraud_detection": ["feature_engineering.py", "scoring_pipeline.py", "alert_generator.py"],
                "regulatory": ["daily_report.py", "monthly_compliance.py", "audit_trail.py"]
            },
            "transformations": ["currency_conversion.py", "risk_scoring.py", "anonymization.py"],
            "quality": ["transaction_checks.py", "balance_reconciliation.py"],
            "utils": ["config.py", "encryption.py", "audit_logger.py"]
        },
        "tests": {
            "unit": ["test_currency.py", "test_risk_scoring.py", "test_anonymization.py"],
            "integration": ["test_transaction_pipeline.py", "test_fraud_pipeline.py"],
            "compliance": ["test_gdpr_requirements.py", "test_sox_controls.py"],
            "fixtures": ["sample_transactions.json", "sample_accounts.json"]
        },
        "config": ["dev.yaml", "staging.yaml", "prod.yaml", "secrets_template.yaml"],
        "migrations": [
            "001_create_raw_transactions.sql",
            "002_create_fraud_scores.sql",
            "003_create_regulatory_views.sql"
        ],
        "docs": ["architecture.md", "runbook.md", "compliance_mapping.md", "disaster_recovery.md"]
    }
}

def print_tree(structure, prefix=""):
    """Print a directory tree from a nested dict."""
    if isinstance(structure, dict):
        items = list(structure.items())
        for i, (name, content) in enumerate(items):
            connector = "└── " if i == len(items) - 1 else "├── "
            print(f"{prefix}{connector}{name}/")
            extension = "    " if i == len(items) - 1 else "│   "
            print_tree(content, prefix + extension)
    elif isinstance(structure, list):
        for i, item in enumerate(structure):
            connector = "└── " if i == len(structure) - 1 else "├── "
            print(f"{prefix}{connector}{item}")

print("📁 FinanceCorps Data Platform:")
print_tree(finance_repo)


---
# 📗 Section 3: Code Organization Patterns

## From Monolith to Modular

The #1 mistake in DE: putting everything in one giant notebook/script.
Modular code is testable, reusable, and deployable.

In [0]:
# ANTI-PATTERN: Monolithic pipeline (everything in one place)
# This is what NOT to do — hard to test, hard to reuse, hard to debug

def monolithic_pipeline_BAD():
    """This is the WRONG way — everything coupled together."""
    # Extract
    raw_df = spark.sql("SELECT * FROM techmart_dw.orders")
    
    # Transform (mixed with business logic, no separation)
    from pyspark.sql.functions import col, when, sum as spark_sum, count
    cleaned = raw_df.filter(col("status") != "cancelled")
    cleaned = cleaned.withColumn("amount_tier",
        when(col("total_amount") > 5000, "high")
        .when(col("total_amount") > 1000, "medium")
        .otherwise("low"))
    
    # Aggregate (can't test this independently)
    result = cleaned.groupBy("amount_tier").agg(
        count("*").alias("order_count"),
        spark_sum("total_amount").alias("total_revenue")
    )
    
    # Load (hardcoded target)
    result.write.mode("overwrite").saveAsTable("gold_order_summary")
    return result

print("❌ ANTI-PATTERN: Monolithic pipeline")
print("   Problems:")
print("   - Can't test transformations independently")
print("   - Can't reuse cleaning logic elsewhere")
print("   - Hardcoded source/target — can't switch environments")
print("   - One failure = entire pipeline fails with no granularity")


In [0]:
# ✅ CORRECT PATTERN: Modular pipeline with separation of concerns

# --- transformations/cleaning.py ---
class OrderCleaner:
    """Reusable order cleaning transformations."""
    
    VALID_STATUSES = ("completed", "shipped", "processing", "pending")
    
    @staticmethod
    def remove_cancelled(df):
        """Remove cancelled orders."""
        from pyspark.sql.functions import col
        return df.filter(col("status").isin(OrderCleaner.VALID_STATUSES))
    
    @staticmethod
    def add_amount_tier(df, high_threshold=5000, medium_threshold=1000):
        """Add amount tier classification."""
        from pyspark.sql.functions import col, when
        return df.withColumn("amount_tier",
            when(col("total_amount") > high_threshold, "high")
            .when(col("total_amount") > medium_threshold, "medium")
            .otherwise("low"))
    
    @staticmethod
    def standardize_columns(df):
        """Ensure consistent column naming."""
        from pyspark.sql.functions import col, lower, trim
        return df.withColumn("status", lower(trim(col("status"))))


# --- transformations/aggregations.py ---
class OrderAggregator:
    """Reusable aggregation functions."""
    
    @staticmethod
    def summarize_by_tier(df):
        """Aggregate orders by amount tier."""
        from pyspark.sql.functions import count, sum as spark_sum, avg as spark_avg
        return df.groupBy("amount_tier").agg(
            count("*").alias("order_count"),
            spark_sum("total_amount").alias("total_revenue"),
            spark_avg("total_amount").alias("avg_order_value")
        )


# --- pipelines/gold/order_summary.py ---
class OrderSummaryPipeline:
    """Orchestrates the order summary pipeline."""
    
    def __init__(self, config):
        self.config = config
        self.cleaner = OrderCleaner()
        self.aggregator = OrderAggregator()
    
    def extract(self):
        """Extract from source."""
        source = self.config["source"]
        return spark.table(f"{source['database']}.{source['table']}")
    
    def transform(self, df):
        """Apply all transformations."""
        df = self.cleaner.remove_cancelled(df)
        df = self.cleaner.add_amount_tier(df)
        return self.aggregator.summarize_by_tier(df)
    
    def load(self, df):
        """Load to target (respects environment config)."""
        target = self.config["target"]
        df.write.mode(target.get("mode", "overwrite")).saveAsTable(
            f"{target['database']}.{target['table']}"
        )
    
    def run(self):
        """Execute full pipeline."""
        print(f"🚀 Running: {self.config.get('name', 'unnamed')}")
        raw = self.extract()
        print(f"   Extracted: {raw.count()} rows")
        transformed = self.transform(raw)
        print(f"   Transformed: {transformed.count()} rows")
        # Don't actually write in this demo
        transformed.show()
        print("   ✅ Pipeline complete")
        return transformed


# Run it with config
config = {
    "name": "order_summary_pipeline",
    "source": {"database": "techmart_dw", "table": "orders"},
    "target": {"database": "techmart_dw", "table": "gold_order_summary", "mode": "overwrite"}
}

pipeline = OrderSummaryPipeline(config)
result = pipeline.run()


In [0]:
# ============================================
# 🎯 YOUR TURN: Refactor a Monolithic Pipeline
# ============================================
# Refactor this monolithic code into modular classes:
#
# MONOLITH (to refactor):
#   df = spark.table("techmart_dw.customers")
#   df = df.filter("lifetime_value > 1000")
#   df = df.withColumn("segment_tier", 
#       when(col("lifetime_value") > 40000, "VIP")
#       .when(col("lifetime_value") > 20000, "Premium")
#       .otherwise("Standard"))
#   result = df.groupBy("segment_tier").agg(count("*"), avg("lifetime_value"))
#   result.write.mode("overwrite").saveAsTable("gold_customer_segments")
#
# Create:
# 1. A CustomerCleaner class with filter and tier methods
# 2. A CustomerAggregator class with a summarize method
# 3. A CustomerSegmentPipeline class that orchestrates them
# 4. Run with a config dict
#
# Write your code below:


In [0]:
# ============================================
# ✅ SOLUTION
# ============================================
from pyspark.sql.functions import col, when, count, avg as spark_avg

class CustomerCleaner:
    """Customer data cleaning transformations."""
    
    @staticmethod
    def filter_active(df, min_ltv=1000):
        """Filter to customers above minimum lifetime value."""
        return df.filter(col("lifetime_value") > min_ltv)
    
    @staticmethod
    def add_segment_tier(df):
        """Classify customers into segment tiers."""
        return df.withColumn("segment_tier",
            when(col("lifetime_value") > 40000, "VIP")
            .when(col("lifetime_value") > 20000, "Premium")
            .otherwise("Standard"))


class CustomerAggregator:
    """Customer aggregation functions."""
    
    @staticmethod
    def summarize_by_segment(df):
        """Aggregate by segment tier."""
        return df.groupBy("segment_tier").agg(
            count("*").alias("customer_count"),
            spark_avg("lifetime_value").alias("avg_ltv")
        )


class CustomerSegmentPipeline:
    """Customer segmentation pipeline."""
    
    def __init__(self, config):
        self.config = config
    
    def run(self):
        # Extract
        src = self.config["source"]
        df = spark.table(f"{src['database']}.{src['table']}")
        print(f"Extracted: {df.count()} customers")
        
        # Transform
        df = CustomerCleaner.filter_active(df, self.config.get("min_ltv", 1000))
        df = CustomerCleaner.add_segment_tier(df)
        result = CustomerAggregator.summarize_by_segment(df)
        
        # Show (don't write in demo)
        result.show()
        return result

# Run
config = {
    "source": {"database": "techmart_dw", "table": "customers"},
    "target": {"database": "techmart_dw", "table": "gold_customer_segments"},
    "min_ltv": 1000
}
pipeline = CustomerSegmentPipeline(config)
pipeline.run()


---
# 📗 Section 4: Environment Management

## Dev / Staging / Production

Every professional data platform has at least 3 environments:

| Environment | Purpose | Data | Who Uses It |
|---|---|---|---|
| **Dev** | Development & experimentation | Sample/synthetic | Individual devs |
| **Staging** | Integration testing | Prod-like (subset) | QA / team |
| **Production** | Live business data | Full dataset | End users |

## Key Principles
1. **Same code, different config** — never hardcode environment-specific values
2. **Least privilege** — dev can't accidentally write to prod
3. **Naming conventions** — `dev_techmart.orders` vs `prod_techmart.orders`
4. **Secrets isolation** — each env has its own credentials

In [0]:
# Environment configuration system
import os
from dataclasses import dataclass, field
from typing import Dict, Any

@dataclass
class EnvironmentConfig:
    """Configuration for a specific environment."""
    name: str
    database_prefix: str
    catalog: str
    storage_path: str
    log_level: str
    batch_size: int
    retry_count: int
    alert_emails: list = field(default_factory=list)
    
    @property
    def database(self):
        return f"{self.database_prefix}_techmart"
    
    def get_table(self, table_name: str) -> str:
        """Get fully qualified table name for this environment."""
        return f"{self.catalog}.{self.database}.{table_name}"


# Define environments
ENVIRONMENTS = {
    "dev": EnvironmentConfig(
        name="dev",
        database_prefix="dev",
        catalog="dev_catalog",
        storage_path="/mnt/dev/data",
        log_level="DEBUG",
        batch_size=1000,
        retry_count=1,
        alert_emails=["dev-team@techmart.com"]
    ),
    "staging": EnvironmentConfig(
        name="staging",
        database_prefix="staging",
        catalog="staging_catalog",
        storage_path="/mnt/staging/data",
        log_level="INFO",
        batch_size=10000,
        retry_count=2,
        alert_emails=["qa-team@techmart.com"]
    ),
    "prod": EnvironmentConfig(
        name="prod",
        database_prefix="prod",
        catalog="prod_catalog",
        storage_path="/Volumes/prod/data/raw",
        log_level="WARNING",
        batch_size=100000,
        retry_count=3,
        alert_emails=["data-team@techmart.com", "oncall@techmart.com"]
    )
}

def get_config(env_name: str = None) -> EnvironmentConfig:
    """Get config for current environment."""
    if env_name is None:
        env_name = os.environ.get("PIPELINE_ENV", "dev")
    if env_name not in ENVIRONMENTS:
        raise ValueError(f"Unknown environment: {env_name}. Valid: {list(ENVIRONMENTS.keys())}")
    return ENVIRONMENTS[env_name]

# Demo: show all environments
print("🌍 Environment Configurations:")
print(f"{'Setting':<20} {'Dev':<25} {'Staging':<25} {'Prod':<25}")
print("-" * 95)
for setting in ["database", "catalog", "storage_path", "log_level", "batch_size", "retry_count"]:
    vals = [str(getattr(ENVIRONMENTS[e], setting)) for e in ["dev", "staging", "prod"]]
    print(f"{setting:<20} {vals[0]:<25} {vals[1]:<25} {vals[2]:<25}")

# Show table resolution
print("
📋 Table name resolution:")
for env_name, config in ENVIRONMENTS.items():
    print(f"  {env_name}: orders → {config.get_table('orders')}")


In [0]:
# YAML-style config files (represented as dicts since we can't read files in CE)
# In production, these would be config/dev.yaml, config/staging.yaml, config/prod.yaml

config_files = {
    "dev.yaml": {
        "environment": "dev",
        "spark": {
            "shuffle_partitions": 8,
            "adaptive_enabled": True,
            "log_level": "DEBUG"
        },
        "sources": {
            "orders": {
                "database": "dev_techmart",
                "table": "orders",
                "sample_fraction": 0.01  # Only 1% of data in dev
            }
        },
        "targets": {
            "gold_sales": {
                "database": "dev_techmart",
                "table": "gold_daily_sales",
                "mode": "overwrite"  # Always overwrite in dev
            }
        },
        "quality": {
            "fail_on_error": False,  # Don't block in dev
            "null_threshold": 0.10   # Relaxed in dev
        }
    },
    "prod.yaml": {
        "environment": "prod",
        "spark": {
            "shuffle_partitions": 200,
            "adaptive_enabled": True,
            "log_level": "WARNING"
        },
        "sources": {
            "orders": {
                "database": "prod_techmart",
                "table": "orders",
                "sample_fraction": 1.0  # Full data in prod
            }
        },
        "targets": {
            "gold_sales": {
                "database": "prod_techmart",
                "table": "gold_daily_sales",
                "mode": "append"  # Never overwrite in prod!
            }
        },
        "quality": {
            "fail_on_error": True,   # Block deployment on quality failure
            "null_threshold": 0.01   # Strict in prod
        }
    }
}

import json
print("📄 Dev Config:")
print(json.dumps(config_files["dev.yaml"], indent=2))
print("
📄 Prod Config (note the differences):")
print(json.dumps(config_files["prod.yaml"], indent=2))

# Highlight critical differences
print("
⚠️  Critical Dev vs Prod Differences:")
print(f"  {'Setting':<30} {'Dev':<20} {'Prod':<20}")
print(f"  {'-'*70}")
print(f"  {'sample_fraction':<30} {'0.01 (1%)':<20} {'1.0 (100%)':<20}")
print(f"  {'write mode':<30} {'overwrite':<20} {'append':<20}")
print(f"  {'fail_on_error':<30} {'False':<20} {'True':<20}")
print(f"  {'null_threshold':<30} {'10%':<20} {'1%':<20}")


In [0]:
# ============================================
# 🎯 YOUR TURN: Build a Config System
# ============================================
# Create a PipelineConfig class that:
# 1. Takes an environment name ("dev", "staging", "prod")
# 2. Has methods:
#    - get_source_table(name) → returns fully qualified table name
#    - get_write_mode() → "overwrite" for dev, "append" for prod
#    - get_batch_size() → 100 for dev, 10000 for staging, 100000 for prod
#    - should_fail_on_error() → False for dev, True for staging/prod
# 3. Test it by creating configs for all 3 environments and printing settings
#
# Write your code below:


In [0]:
# ============================================
# ✅ SOLUTION
# ============================================
class PipelineConfig:
    """Environment-aware pipeline configuration."""
    
    ENV_SETTINGS = {
        "dev": {"prefix": "dev", "write_mode": "overwrite", "batch_size": 100, "fail_on_error": False},
        "staging": {"prefix": "staging", "write_mode": "append", "batch_size": 10000, "fail_on_error": True},
        "prod": {"prefix": "prod", "write_mode": "append", "batch_size": 100000, "fail_on_error": True}
    }
    
    def __init__(self, environment: str):
        if environment not in self.ENV_SETTINGS:
            raise ValueError(f"Invalid env: {environment}")
        self.env = environment
        self.settings = self.ENV_SETTINGS[environment]
    
    def get_source_table(self, name: str) -> str:
        """Get fully qualified table name."""
        return f"{self.settings['prefix']}_techmart.{name}"
    
    def get_write_mode(self) -> str:
        return self.settings["write_mode"]
    
    def get_batch_size(self) -> int:
        return self.settings["batch_size"]
    
    def should_fail_on_error(self) -> bool:
        return self.settings["fail_on_error"]

# Test all environments
print(f"{'Environment':<12} {'orders table':<30} {'write_mode':<12} {'batch_size':<12} {'fail_on_error'}")
print("-" * 80)
for env in ["dev", "staging", "prod"]:
    cfg = PipelineConfig(env)
    print(f"{env:<12} {cfg.get_source_table('orders'):<30} {cfg.get_write_mode():<12} {cfg.get_batch_size():<12} {cfg.should_fail_on_error()}")


---
# 📗 Section 5: Testing Strategy for Pipelines

## The Testing Pyramid for Data Engineering

```
        /\
       /  \        E2E Tests (few, slow, expensive)
      /    \       - Full pipeline run with real-like data
     /------\
    /        \     Integration Tests (moderate)
   /          \    - Multiple components together
  /            \   - Schema validation
 /--------------\
/                \  Unit Tests (many, fast, cheap)
/                  \ - Individual transform functions
/                    \ - Business logic validation
```

| Test Type | What | Speed | When to Run |
|---|---|---|---|
| Unit | Individual functions | < 1 sec | Every commit |
| Integration | Component interactions | < 1 min | Every PR |
| Contract | Schema compatibility | < 30 sec | Every PR |
| Data Quality | Row-level expectations | 1-5 min | Every pipeline run |
| E2E | Full pipeline | 5-30 min | Before deploy |

In [0]:
# Unit Testing: Test individual transformation functions
# In production, these would use pytest. Here we use assert-based patterns.

class TestRunner:
    """Simple test runner for notebook demonstrations."""
    
    def __init__(self, suite_name):
        self.suite_name = suite_name
        self.passed = 0
        self.failed = 0
        self.errors = []
    
    def test(self, name, func):
        """Run a single test."""
        try:
            func()
            self.passed += 1
            print(f"  ✅ {name}")
        except AssertionError as e:
            self.failed += 1
            self.errors.append((name, str(e)))
            print(f"  ❌ {name}: {e}")
        except Exception as e:
            self.failed += 1
            self.errors.append((name, str(e)))
            print(f"  💥 {name}: {type(e).__name__}: {e}")
    
    def summary(self):
        total = self.passed + self.failed
        print(f"
{'='*50}")
        print(f"  {self.suite_name}: {self.passed}/{total} passed")
        if self.errors:
            print(f"  Failures:")
            for name, err in self.errors:
                print(f"    - {name}: {err}")
        print(f"{'='*50}")
        return self.failed == 0


# --- Functions to test ---
def classify_amount(amount):
    """Classify order amount into tiers."""
    if amount > 5000:
        return "high"
    elif amount > 1000:
        return "medium"
    else:
        return "low"

def clean_email(email):
    """Standardize email address."""
    if not email:
        return None
    return email.strip().lower()

def calculate_discount(amount, tier):
    """Calculate discount based on amount and customer tier."""
    base_rates = {"VIP": 0.15, "Premium": 0.10, "Standard": 0.05}
    rate = base_rates.get(tier, 0.0)
    return round(amount * rate, 2)


# --- Unit Tests ---
runner = TestRunner("Transformation Unit Tests")

# Test classify_amount
runner.test("classify_amount: high tier", lambda: assert_equal(classify_amount(6000), "high"))
runner.test("classify_amount: medium tier", lambda: assert_equal(classify_amount(2500), "medium"))
runner.test("classify_amount: low tier", lambda: assert_equal(classify_amount(500), "low"))
runner.test("classify_amount: boundary high", lambda: assert_equal(classify_amount(5001), "high"))
runner.test("classify_amount: boundary medium", lambda: assert_equal(classify_amount(1001), "medium"))

# Test clean_email
runner.test("clean_email: strips and lowers", lambda: assert_equal(clean_email("  JOHN@GMAIL.COM  "), "john@gmail.com"))
runner.test("clean_email: handles None", lambda: assert_equal(clean_email(None), None))
runner.test("clean_email: handles empty", lambda: assert_equal(clean_email(""), None))

# Test calculate_discount
runner.test("discount: VIP rate", lambda: assert_equal(calculate_discount(1000, "VIP"), 150.0))
runner.test("discount: Premium rate", lambda: assert_equal(calculate_discount(1000, "Premium"), 100.0))
runner.test("discount: unknown tier", lambda: assert_equal(calculate_discount(1000, "Unknown"), 0.0))

runner.summary()


In [0]:
# Helper function for assertions (define before tests use it)
def assert_equal(actual, expected, msg=""):
    """Assert two values are equal."""
    if actual != expected:
        raise AssertionError(f"Expected {expected}, got {actual}. {msg}")

def assert_true(condition, msg=""):
    """Assert condition is True."""
    if not condition:
        raise AssertionError(f"Condition was False. {msg}")

def assert_in(item, collection, msg=""):
    """Assert item is in collection."""
    if item not in collection:
        raise AssertionError(f"{item} not found in {collection}. {msg}")

# Integration Test: Test with actual Spark DataFrames
from pyspark.sql.functions import col, when, lower, trim

def test_order_cleaning_integration():
    """Integration test: verify cleaning pipeline on real data."""
    runner = TestRunner("Order Cleaning Integration Tests")
    
    # Get sample data
    df = spark.table("techmart_dw.orders").limit(100)
    
    # Test: filter removes cancelled orders
    def test_filter_cancelled():
        cleaned = df.filter(col("status") != "cancelled")
        cancelled_count = cleaned.filter(col("status") == "cancelled").count()
        assert_equal(cancelled_count, 0, "Should have no cancelled orders after filter")
    
    # Test: amount tier assignment
    def test_amount_tiers():
        with_tier = df.withColumn("tier",
            when(col("total_amount") > 5000, "high")
            .when(col("total_amount") > 1000, "medium")
            .otherwise("low"))
        
        # Verify no nulls in tier
        null_tiers = with_tier.filter(col("tier").isNull()).count()
        assert_equal(null_tiers, 0, "All rows should have a tier")
        
        # Verify tier values are valid
        distinct_tiers = set(row.tier for row in with_tier.select("tier").distinct().collect())
        for tier in distinct_tiers:
            assert_in(tier, ["high", "medium", "low"], f"Invalid tier: {tier}")
    
    # Test: row count preserved (no data loss)
    def test_no_data_loss():
        original_count = df.count()
        # Apply non-filtering transforms
        transformed = df.withColumn("tier",
            when(col("total_amount") > 5000, "high")
            .when(col("total_amount") > 1000, "medium")
            .otherwise("low"))
        assert_equal(transformed.count(), original_count, "Transform should not lose rows")
    
    runner.test("Filter removes cancelled", test_filter_cancelled)
    runner.test("Amount tiers assigned correctly", test_amount_tiers)
    runner.test("No data loss in transforms", test_no_data_loss)
    runner.summary()

test_order_cleaning_integration()


In [0]:
# Contract Tests: Schema validation
def test_schema_contract():
    """Verify table schemas match expected contracts."""
    runner = TestRunner("Schema Contract Tests")
    
    # Define expected schemas (contracts)
    expected_schemas = {
        "orders": {
            "required_columns": ["order_id", "customer_id", "total_amount", "status", "order_date"],
            "column_types": {
                "order_id": "IntegerType()" if False else None,  # flexible
                "total_amount": "DecimalType",  # partial match
            },
            "min_rows": 10000
        },
        "customers": {
            "required_columns": ["customer_id", "email", "customer_segment", "lifetime_value"],
            "column_types": {},
            "min_rows": 5000
        }
    }
    
    for table_name, contract in expected_schemas.items():
        df = spark.table(f"techmart_dw.{table_name}")
        actual_columns = set(df.columns)
        
        # Test: all required columns exist
        def make_col_test(tbl, req_cols, act_cols):
            def test_fn():
                missing = set(req_cols) - act_cols
                assert_equal(len(missing), 0, f"Missing columns: {missing}")
            return test_fn
        
        runner.test(
            f"{table_name}: has required columns",
            make_col_test(table_name, contract["required_columns"], actual_columns)
        )
        
        # Test: minimum row count
        def make_count_test(tbl, min_rows):
            def test_fn():
                count = spark.table(f"techmart_dw.{tbl}").count()
                assert_true(count >= min_rows, f"Expected >= {min_rows}, got {count}")
            return test_fn
        
        runner.test(
            f"{table_name}: meets minimum row count ({contract['min_rows']})",
            make_count_test(table_name, contract["min_rows"])
        )
    
    runner.summary()

test_schema_contract()


In [0]:
# ============================================
# 🎯 YOUR TURN: Write a Test Suite
# ============================================
# Write a complete test suite for a "customer enrichment" function:
#
# Function to test:
#   def enrich_customer(df):
#       - Adds "value_tier" column (based on lifetime_value thresholds)
#       - Adds "tenure_years" column (years since signup)
#       - Filters out customers with NULL email
#       - Returns enriched DataFrame
#
# Write tests for:
# 1. value_tier is correctly assigned (test boundaries)
# 2. tenure_years is non-negative
# 3. No NULL emails in output
# 4. Output has expected columns
# 5. Row count <= input count (filtering happened)
#
# Use the TestRunner class from above.
#
# Write your code below:


In [0]:
# ============================================
# ✅ SOLUTION
# ============================================
from pyspark.sql.functions import col, when, year, current_date, datediff

def enrich_customer(df):
    """Enrich customer data with derived columns."""
    enriched = df.filter(col("email").isNotNull())
    enriched = enriched.withColumn("value_tier",
        when(col("lifetime_value") > 40000, "platinum")
        .when(col("lifetime_value") > 20000, "gold")
        .when(col("lifetime_value") > 5000, "silver")
        .otherwise("bronze"))
    enriched = enriched.withColumn("tenure_years",
        (datediff(current_date(), col("signup_date")) / 365).cast("int"))
    return enriched

# Test suite
runner = TestRunner("Customer Enrichment Tests")
df = spark.table("techmart_dw.customers").limit(200)
enriched = enrich_customer(df)

def test_value_tier_assigned():
    null_tiers = enriched.filter(col("value_tier").isNull()).count()
    assert_equal(null_tiers, 0, "All rows should have value_tier")

def test_valid_tier_values():
    tiers = set(r.value_tier for r in enriched.select("value_tier").distinct().collect())
    valid = {"platinum", "gold", "silver", "bronze"}
    invalid = tiers - valid
    assert_equal(len(invalid), 0, f"Invalid tiers found: {invalid}")

def test_tenure_non_negative():
    negative = enriched.filter(col("tenure_years") < 0).count()
    assert_equal(negative, 0, "Tenure should never be negative")

def test_no_null_emails():
    nulls = enriched.filter(col("email").isNull()).count()
    assert_equal(nulls, 0, "No NULL emails should remain")

def test_has_expected_columns():
    required = {"value_tier", "tenure_years", "customer_id", "email"}
    actual = set(enriched.columns)
    missing = required - actual
    assert_equal(len(missing), 0, f"Missing columns: {missing}")

def test_row_count_reduced():
    assert_true(enriched.count() <= df.count(), "Filtering should reduce or maintain count")

runner.test("Value tier assigned to all rows", test_value_tier_assigned)
runner.test("Only valid tier values", test_valid_tier_values)
runner.test("Tenure is non-negative", test_tenure_non_negative)
runner.test("No NULL emails in output", test_no_null_emails)
runner.test("Has expected columns", test_has_expected_columns)
runner.test("Row count reduced by filter", test_row_count_reduced)
runner.summary()


---
# 📗 Section 6: CI Pipeline Design

## What Happens on Every Commit

```
┌─────────────┐    ┌──────────┐    ┌─────────────┐    ┌──────────────┐    ┌─────────┐
│  Developer  │───▶│  Commit  │───▶│   Linting   │───▶│  Unit Tests  │───▶│  Build  │
│  pushes     │    │  to Git  │    │  (ruff/black)│    │  (pytest)    │    │ artifact│
└─────────────┘    └──────────┘    └─────────────┘    └──────────────┘    └─────────┘
                                          │                   │                  │
                                          ▼                   ▼                  ▼
                                   ┌─────────────┐    ┌──────────────┐    ┌─────────┐
                                   │  Fail fast  │    │ Integration  │    │  Schema │
                                   │  on style   │    │   Tests      │    │  Check  │
                                   └─────────────┘    └──────────────┘    └─────────┘
```

## GitHub Actions CI Pipeline (Conceptual)

```yaml
# .github/workflows/ci.yml
name: Data Pipeline CI
on:
  push:
    branches: [main, develop]
  pull_request:
    branches: [main]

jobs:
  lint:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - run: pip install ruff black
      - run: ruff check src/
      - run: black --check src/

  unit-tests:
    runs-on: ubuntu-latest
    needs: lint
    steps:
      - uses: actions/checkout@v4
      - run: pip install -e ".[test]"
      - run: pytest tests/unit/ -v --cov=src/

  integration-tests:
    runs-on: ubuntu-latest
    needs: unit-tests
    steps:
      - uses: actions/checkout@v4
      - run: pip install -e ".[test]"
      - run: pytest tests/integration/ -v

  schema-validation:
    runs-on: ubuntu-latest
    needs: unit-tests
    steps:
      - uses: actions/checkout@v4
      - run: python scripts/validate_schemas.py
```

In [0]:
# Implement a CI pipeline simulator in Python
from datetime import datetime
import time

class CIStage:
    """Represents a single CI pipeline stage."""
    
    def __init__(self, name, check_func, depends_on=None):
        self.name = name
        self.check_func = check_func
        self.depends_on = depends_on or []
        self.status = "pending"
        self.duration = 0
        self.output = ""
    
    def run(self):
        """Execute this stage."""
        start = time.time()
        try:
            self.output = self.check_func()
            self.status = "passed"
        except Exception as e:
            self.status = "failed"
            self.output = str(e)
        self.duration = time.time() - start
        return self.status == "passed"


class CIPipeline:
    """Simulates a CI pipeline execution."""
    
    def __init__(self, name):
        self.name = name
        self.stages = []
        self.start_time = None
    
    def add_stage(self, name, check_func, depends_on=None):
        self.stages.append(CIStage(name, check_func, depends_on))
    
    def run(self):
        """Execute all stages in order, stopping on failure."""
        self.start_time = datetime.now()
        print(f"🚀 CI Pipeline: {self.name}")
        print(f"   Started: {self.start_time.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"{'='*60}")
        
        all_passed = True
        for stage in self.stages:
            # Check dependencies
            dep_failed = [s for s in self.stages 
                         if s.name in stage.depends_on and s.status == "failed"]
            if dep_failed:
                stage.status = "skipped"
                print(f"  ⏭️  {stage.name}: SKIPPED (dependency failed)")
                continue
            
            success = stage.run()
            icon = "✅" if success else "❌"
            print(f"  {icon} {stage.name}: {stage.status.upper()} ({stage.duration:.2f}s)")
            if stage.output:
                for line in stage.output.split("\n")[:3]:
                    print(f"      {line}")
            
            if not success:
                all_passed = False
                print(f"
  🛑 Pipeline FAILED at stage: {stage.name}")
                break
        
        print(f"{'='*60}")
        total_time = sum(s.duration for s in self.stages)
        status = "✅ PASSED" if all_passed else "❌ FAILED"
        print(f"  Result: {status} | Total time: {total_time:.2f}s")
        return all_passed


# Define CI checks for our TechMart pipeline
def lint_check():
    """Simulate code linting."""
    # Check for common issues in our pipeline code
    issues = []
    # Simulate checking for hardcoded values
    sample_code = "spark.table('prod_techmart.orders')"  # BAD: hardcoded env
    if "prod_" in sample_code:
        issues.append("WARNING: Hardcoded production reference found")
    if not issues:
        return "All files pass linting (ruff + black)"
    return "\n".join(issues)

def unit_test_check():
    """Run unit tests."""
    # Actually test our functions
    assert classify_amount(6000) == "high"
    assert classify_amount(2500) == "medium"
    assert classify_amount(500) == "low"
    assert clean_email("  TEST@GMAIL.COM  ") == "test@gmail.com"
    assert clean_email(None) is None
    return "12 tests passed, 0 failed"

def integration_test_check():
    """Run integration tests with Spark."""
    df = spark.table("techmart_dw.orders").limit(50)
    assert df.count() == 50, "Failed to read orders"
    # Test transformation
    from pyspark.sql.functions import col, when
    result = df.withColumn("tier", when(col("total_amount") > 5000, "high").otherwise("other"))
    assert "tier" in result.columns, "Tier column not added"
    return "3 integration tests passed"

def schema_validation_check():
    """Validate schemas match contracts."""
    required = {"order_id", "customer_id", "total_amount", "status"}
    actual = set(spark.table("techmart_dw.orders").columns)
    missing = required - actual
    if missing:
        raise AssertionError(f"Missing columns: {missing}")
    return f"Schema valid: {len(required)} required columns present"

# Build and run the pipeline
ci = CIPipeline("techmart-data-platform / PR #42: Add customer tier logic")
ci.add_stage("Lint & Format", lint_check)
ci.add_stage("Unit Tests", unit_test_check, depends_on=["Lint & Format"])
ci.add_stage("Integration Tests", integration_test_check, depends_on=["Unit Tests"])
ci.add_stage("Schema Validation", schema_validation_check, depends_on=["Unit Tests"])
ci.run()


In [0]:
# ============================================
# 🎯 YOUR TURN: Design a CI Pipeline
# ============================================
# Create a CIPipeline for a "customer_360" feature that includes:
# 1. "Code Quality" stage — check that all functions have docstrings
# 2. "Unit Tests" stage — test a customer scoring function
# 3. "Data Quality" stage — verify customers table has no NULL customer_ids
# 4. "Schema Check" stage — verify output would have required columns
#
# Define the check functions and wire them into a CIPipeline.
# Make stages 2-4 depend on stage 1.
#
# Write your code below:


In [0]:
# ============================================
# ✅ SOLUTION
# ============================================
def code_quality_check():
    """Check code quality standards."""
    # Simulate checking functions have docstrings
    functions_to_check = [classify_amount, clean_email, calculate_discount]
    missing_docs = [f.__name__ for f in functions_to_check if not f.__doc__]
    if missing_docs:
        raise AssertionError(f"Missing docstrings: {missing_docs}")
    return f"All {len(functions_to_check)} functions have docstrings"

def customer_scoring_test():
    """Test customer scoring logic."""
    def score_customer(ltv, tenure_years):
        score = 0
        if ltv > 40000: score += 50
        elif ltv > 20000: score += 30
        elif ltv > 5000: score += 15
        if tenure_years > 5: score += 30
        elif tenure_years > 2: score += 15
        return min(score, 100)
    
    assert score_customer(50000, 6) == 80, "VIP + long tenure"
    assert score_customer(25000, 3) == 45, "Gold + medium tenure"
    assert score_customer(1000, 1) == 0, "Low value + new"
    return "Customer scoring: 3/3 tests passed"

def data_quality_check():
    """Check for NULL primary keys."""
    null_count = spark.sql(
        "SELECT COUNT(*) as cnt FROM techmart_dw.customers WHERE customer_id IS NULL"
    ).collect()[0].cnt
    if null_count > 0:
        raise AssertionError(f"Found {null_count} NULL customer_ids!")
    return "Zero NULL primary keys ✓"

def output_schema_check():
    """Verify expected output columns."""
    required_output = {"customer_id", "email", "lifetime_value", "customer_segment", "value_tier", "tenure_years"}
    # Simulate what our pipeline would produce
    df = spark.table("techmart_dw.customers")
    available = set(df.columns)
    # Check base columns exist (tier/tenure added by transform)
    base_required = {"customer_id", "email", "lifetime_value", "customer_segment"}
    missing = base_required - available
    if missing:
        raise AssertionError(f"Missing base columns: {missing}")
    return f"Schema check passed: {len(base_required)} base columns verified"

# Build pipeline
ci = CIPipeline("customer-360 / PR #15: Add scoring model")
ci.add_stage("Code Quality", code_quality_check)
ci.add_stage("Unit Tests", customer_scoring_test, depends_on=["Code Quality"])
ci.add_stage("Data Quality", data_quality_check, depends_on=["Code Quality"])
ci.add_stage("Schema Check", output_schema_check, depends_on=["Code Quality"])
ci.run()


---
# 📗 Section 7: CD Pipeline Design (Deployment Strategies)

## Deployment Flow

```
┌──────────┐    ┌──────────┐    ┌────────────┐    ┌──────────┐    ┌──────────┐
│  CI Pass │───▶│  Deploy  │───▶│   Smoke    │───▶│ Promote  │───▶│  Monitor │
│          │    │  Staging │    │   Tests    │    │  to Prod │    │  & Alert │
└──────────┘    └──────────┘    └────────────┘    └──────────┘    └──────────┘
                                       │                                │
                                       ▼                                ▼
                                ┌────────────┐                   ┌──────────┐
                                │  Rollback  │◀──────────────────│  Failure │
                                │  if fail   │                   │ Detected │
                                └────────────┘                   └──────────┘
```

## Deployment Strategies for Data Pipelines

| Strategy | How it Works | Best For |
|---|---|---|
| **Blue-Green** | Run old + new in parallel, switch traffic | Stateless transforms |
| **Canary** | Process small subset with new code first | High-risk changes |
| **Rolling** | Update one component at a time | Multi-stage pipelines |
| **Shadow** | Run new code alongside, compare outputs | ML model updates |

In [0]:
# Deployment Simulator
from datetime import datetime
from enum import Enum

class DeployStatus(Enum):
    PENDING = "pending"
    DEPLOYING = "deploying"
    SMOKE_TESTING = "smoke_testing"
    DEPLOYED = "deployed"
    ROLLING_BACK = "rolling_back"
    ROLLED_BACK = "rolled_back"
    FAILED = "failed"


class DeploymentManager:
    """Simulates pipeline deployment with rollback capability."""
    
    def __init__(self, pipeline_name, environment):
        self.pipeline_name = pipeline_name
        self.environment = environment
        self.status = DeployStatus.PENDING
        self.history = []
        self.current_version = "v1.2.0"
        self.deploying_version = None
    
    def _log(self, message):
        entry = {"timestamp": datetime.now().isoformat(), "message": message, "status": self.status.value}
        self.history.append(entry)
        print(f"  [{self.status.value:>14}] {message}")
    
    def deploy(self, new_version, changes, smoke_tests):
        """Deploy a new version with smoke tests."""
        self.deploying_version = new_version
        print(f"
{'='*60}")
        print(f"🚀 Deploying {self.pipeline_name}")
        print(f"   {self.current_version} → {new_version}")
        print(f"   Environment: {self.environment}")
        print(f"   Changes: {', '.join(changes)}")
        print(f"{'='*60}")
        
        # Step 1: Pre-deployment validation
        self.status = DeployStatus.DEPLOYING
        self._log(f"Starting deployment of {new_version}")
        self._log("Validating deployment package...")
        self._log("Checking environment compatibility...")
        
        # Step 2: Deploy to target
        self._log(f"Deploying to {self.environment}...")
        self._log("Updating pipeline configuration...")
        self._log("Restarting pipeline workers...")
        
        # Step 3: Smoke tests
        self.status = DeployStatus.SMOKE_TESTING
        self._log("Running smoke tests...")
        
        all_passed = True
        for test_name, test_func in smoke_tests.items():
            try:
                test_func()
                self._log(f"  ✅ Smoke test passed: {test_name}")
            except Exception as e:
                self._log(f"  ❌ Smoke test FAILED: {test_name} — {e}")
                all_passed = False
                break
        
        if all_passed:
            self.status = DeployStatus.DEPLOYED
            self.current_version = new_version
            self._log(f"✅ Deployment successful! Now running {new_version}")
        else:
            self._log("⚠️ Smoke tests failed! Initiating rollback...")
            self.rollback()
        
        return all_passed
    
    def rollback(self):
        """Rollback to previous version."""
        self.status = DeployStatus.ROLLING_BACK
        self._log(f"Rolling back from {self.deploying_version} to {self.current_version}")
        self._log("Restoring previous configuration...")
        self._log("Restarting with previous version...")
        self.status = DeployStatus.ROLLED_BACK
        self._log(f"✅ Rollback complete. Running {self.current_version}")


# Demo: Successful deployment
deployer = DeploymentManager("orders_pipeline", "staging")

smoke_tests_pass = {
    "table_exists": lambda: assert_true(
        spark.sql("SHOW TABLES IN techmart_dw LIKE 'orders'").count() > 0
    ),
    "data_fresh": lambda: assert_true(
        spark.table("techmart_dw.orders").count() > 0
    ),
    "schema_valid": lambda: assert_true(
        "order_id" in spark.table("techmart_dw.orders").columns
    )
}

deployer.deploy(
    "v1.3.0",
    ["Added customer tier logic", "Updated aggregation window"],
    smoke_tests_pass
)


In [0]:
# Demo: Failed deployment with automatic rollback
deployer2 = DeploymentManager("customer_360_pipeline", "staging")

smoke_tests_fail = {
    "table_exists": lambda: assert_true(True),
    "schema_valid": lambda: assert_true(True),
    "data_quality": lambda: (_ for _ in ()).throw(AssertionError("NULL rate 15% exceeds threshold 5%"))
}

# Fix the lambda to actually raise
def failing_quality_check():
    raise AssertionError("NULL rate 15% exceeds threshold 5%")

smoke_tests_fail["data_quality"] = failing_quality_check

deployer2.deploy(
    "v2.0.0",
    ["Major refactor of customer scoring", "New data source added"],
    smoke_tests_fail
)

print("
📋 Deployment History:")
for entry in deployer2.history:
    print(f"  {entry['timestamp'][:19]} | {entry['status']:>14} | {entry['message']}")


---
# 📗 Section 8: Database Migrations

## Schema Changes as Code

Database migrations ensure schema changes are:
- **Versioned** — tracked in Git like any other code
- **Ordered** — applied in sequence
- **Repeatable** — same result every time
- **Reversible** — can be rolled back (when possible)

## Migration Naming Convention
```
migrations/
├── 001_create_bronze_tables.sql
├── 002_add_customer_segment_column.sql
├── 003_create_silver_views.sql
├── 004_add_index_on_order_date.sql
└── 005_rename_column_amt_to_amount.sql
```

In [0]:
# Migration system implementation
from datetime import datetime

class Migration:
    """Represents a single database migration."""
    
    def __init__(self, version, name, up_sql, down_sql=None, description=""):
        self.version = version
        self.name = name
        self.up_sql = up_sql
        self.down_sql = down_sql
        self.description = description
        self.applied_at = None
    
    def __repr__(self):
        status = f"applied {self.applied_at}" if self.applied_at else "pending"
        return f"Migration({self.version}: {self.name} [{status}])"


class MigrationRunner:
    """Manages and executes database migrations."""
    
    def __init__(self, database):
        self.database = database
        self.migrations = []
        self.applied = []
    
    def add(self, version, name, up_sql, down_sql=None, description=""):
        """Register a migration."""
        self.migrations.append(Migration(version, name, up_sql, down_sql, description))
        self.migrations.sort(key=lambda m: m.version)
    
    def get_pending(self):
        """Get migrations not yet applied."""
        applied_versions = {m.version for m in self.applied}
        return [m for m in self.migrations if m.version not in applied_versions]
    
    def apply(self, dry_run=False):
        """Apply all pending migrations."""
        pending = self.get_pending()
        if not pending:
            print("  ✅ No pending migrations")
            return True
        
        print(f"  📋 {len(pending)} pending migration(s):")
        for m in pending:
            print(f"     {m.version}: {m.name}")
        print()
        
        for migration in pending:
            print(f"  ▶️  Applying {migration.version}: {migration.name}")
            if migration.description:
                print(f"     Description: {migration.description}")
            
            if dry_run:
                print(f"     [DRY RUN] Would execute:")
                for line in migration.up_sql.strip().split("\n")[:5]:
                    print(f"       {line}")
                if len(migration.up_sql.strip().split("\n")) > 5:
                    print(f"       ... ({len(migration.up_sql.strip().split(chr(10)))} lines total)")
            else:
                try:
                    # Execute each statement
                    for stmt in migration.up_sql.split(";"):
                        stmt = stmt.strip()
                        if stmt:
                            spark.sql(stmt)
                    migration.applied_at = datetime.now().isoformat()
                    self.applied.append(migration)
                    print(f"     ✅ Applied successfully")
                except Exception as e:
                    print(f"     ❌ FAILED: {e}")
                    print(f"     🛑 Stopping migration. Manual intervention required.")
                    return False
        
        return True
    
    def rollback(self, steps=1):
        """Rollback the last N migrations."""
        if not self.applied:
            print("  ⚠️ Nothing to rollback")
            return
        
        to_rollback = self.applied[-steps:]
        for migration in reversed(to_rollback):
            if migration.down_sql:
                print(f"  ⏪ Rolling back {migration.version}: {migration.name}")
                try:
                    for stmt in migration.down_sql.split(";"):
                        stmt = stmt.strip()
                        if stmt:
                            spark.sql(stmt)
                    self.applied.remove(migration)
                    migration.applied_at = None
                    print(f"     ✅ Rolled back")
                except Exception as e:
                    print(f"     ❌ Rollback failed: {e}")
            else:
                print(f"  ⚠️ {migration.version} has no rollback SQL (forward-only)")
    
    def status(self):
        """Show migration status."""
        print(f"\n📊 Migration Status for {self.database}:")
        print(f"{'Version':<10} {'Name':<40} {'Status':<20}")
        print("-" * 70)
        applied_versions = {m.version for m in self.applied}
        for m in self.migrations:
            if m.version in applied_versions:
                print(f"{m.version:<10} {m.name:<40} ✅ Applied")
            else:
                print(f"{m.version:<10} {m.name:<40} ⏳ Pending")


# Create migration runner for TechMart
runner = MigrationRunner("techmart_dw")

# Define migrations
runner.add("001", "create_migration_tracking",
    up_sql="""
        CREATE TABLE IF NOT EXISTS techmart_dw.schema_migrations (
            version STRING,
            name STRING,
            applied_at TIMESTAMP,
            checksum STRING
        )
    """,
    down_sql="DROP TABLE IF EXISTS techmart_dw.schema_migrations",
    description="Track which migrations have been applied"
)

runner.add("002", "add_customer_loyalty_tier",
    up_sql="""
        ALTER TABLE techmart_dw.customers ADD COLUMNS (
            loyalty_tier STRING COMMENT 'Calculated loyalty tier'
        )
    """,
    down_sql="ALTER TABLE techmart_dw.customers DROP COLUMN loyalty_tier",
    description="Add loyalty tier column for new segmentation feature"
)

runner.add("003", "create_gold_daily_metrics",
    up_sql="""
        CREATE TABLE IF NOT EXISTS techmart_dw.gold_daily_metrics (
            metric_date DATE,
            total_orders INT,
            total_revenue DECIMAL(12,2),
            avg_order_value DECIMAL(10,2),
            unique_customers INT,
            _created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
        )
    """,
    down_sql="DROP TABLE IF EXISTS techmart_dw.gold_daily_metrics",
    description="Gold layer daily metrics aggregate table"
)

# Show status and apply
runner.status()
print()
runner.apply(dry_run=True)  # Dry run first to show what would happen


In [0]:
# Actually apply migrations (not dry run)
print("🚀 Applying migrations for real:")
runner.apply(dry_run=False)
runner.status()


In [0]:
# ============================================
# 🎯 YOUR TURN: Write Migration Scripts
# ============================================
# Create a MigrationRunner and add 3 migrations for a new feature:
# "Customer Lifetime Value Recalculation"
#
# Migration 004: Add a "ltv_recalculated_at" TIMESTAMP column to customers
# Migration 005: Create a "techmart_dw.ltv_history" table with columns:
#   customer_id INT, old_ltv DECIMAL(12,2), new_ltv DECIMAL(12,2), 
#   change_reason STRING, calculated_at TIMESTAMP
# Migration 006: Create a view "techmart_dw.v_ltv_changes" that joins
#   customers with ltv_history
#
# Show status, then apply with dry_run=True
#
# Write your code below:


In [0]:
# ============================================
# ✅ SOLUTION
# ============================================
ltv_runner = MigrationRunner("techmart_dw")

ltv_runner.add("004", "add_ltv_recalculated_at",
    up_sql="ALTER TABLE techmart_dw.customers ADD COLUMNS (ltv_recalculated_at TIMESTAMP)",
    down_sql="ALTER TABLE techmart_dw.customers DROP COLUMN ltv_recalculated_at",
    description="Track when LTV was last recalculated"
)

ltv_runner.add("005", "create_ltv_history",
    up_sql="""
        CREATE TABLE IF NOT EXISTS techmart_dw.ltv_history (
            customer_id INT,
            old_ltv DECIMAL(12,2),
            new_ltv DECIMAL(12,2),
            change_reason STRING,
            calculated_at TIMESTAMP
        )
    """,
    down_sql="DROP TABLE IF EXISTS techmart_dw.ltv_history",
    description="Historical record of LTV changes for audit"
)

ltv_runner.add("006", "create_ltv_changes_view",
    up_sql="""
        CREATE OR REPLACE VIEW techmart_dw.v_ltv_changes AS
        SELECT 
            c.customer_id,
            c.email,
            c.lifetime_value AS current_ltv,
            h.old_ltv,
            h.new_ltv,
            h.change_reason,
            h.calculated_at
        FROM techmart_dw.customers c
        LEFT JOIN techmart_dw.ltv_history h ON c.customer_id = h.customer_id
    """,
    down_sql="DROP VIEW IF EXISTS techmart_dw.v_ltv_changes",
    description="View joining customers with their LTV change history"
)

ltv_runner.status()
print()
ltv_runner.apply(dry_run=True)


---
# 📗 Section 9: Feature Flags for Pipelines

## Why Feature Flags in Data Engineering?

- **Gradual rollout** — enable new logic for 10% of data first
- **Kill switch** — instantly disable broken features without redeploying
- **A/B testing** — compare old vs new transformation logic
- **Trunk-based development** — merge incomplete features behind flags

## Feature Flag Patterns
```
if feature_flags.is_enabled("new_scoring_model"):
    result = new_scoring_pipeline(df)
else:
    result = legacy_scoring_pipeline(df)
```

In [0]:
# Feature Flag System for Data Pipelines
from datetime import datetime, date

class FeatureFlag:
    """A single feature flag with metadata."""
    
    def __init__(self, name, enabled=False, rollout_pct=100, 
                 description="", owner="", expires=None):
        self.name = name
        self.enabled = enabled
        self.rollout_pct = rollout_pct  # 0-100
        self.description = description
        self.owner = owner
        self.expires = expires
        self.created_at = datetime.now()
    
    def is_active(self, entity_id=None):
        """Check if flag is active, optionally for a specific entity."""
        if not self.enabled:
            return False
        if self.expires and date.today() > self.expires:
            return False
        if self.rollout_pct < 100 and entity_id is not None:
            # Deterministic: same entity always gets same result
            return (hash(f"{self.name}:{entity_id}") % 100) < self.rollout_pct
        return self.rollout_pct == 100


class FeatureFlagManager:
    """Manages feature flags for pipeline execution."""
    
    def __init__(self):
        self.flags = {}
    
    def register(self, name, **kwargs):
        """Register a new feature flag."""
        self.flags[name] = FeatureFlag(name, **kwargs)
    
    def is_enabled(self, name, entity_id=None):
        """Check if a feature is enabled."""
        flag = self.flags.get(name)
        if flag is None:
            return False  # Unknown flags default to disabled
        return flag.is_active(entity_id)
    
    def enable(self, name):
        """Enable a flag."""
        if name in self.flags:
            self.flags[name].enabled = True
    
    def disable(self, name):
        """Disable a flag (kill switch)."""
        if name in self.flags:
            self.flags[name].enabled = False
    
    def set_rollout(self, name, pct):
        """Set rollout percentage."""
        if name in self.flags:
            self.flags[name].rollout_pct = pct
    
    def status(self):
        """Show all flags and their status."""
        print(f"{'Flag':<35} {'Enabled':<10} {'Rollout':<10} {'Owner':<15} {'Description'}")
        print("-" * 100)
        for name, flag in self.flags.items():
            status = "✅ ON" if flag.enabled else "❌ OFF"
            print(f"{name:<35} {status:<10} {flag.rollout_pct}%{'':>6} {flag.owner:<15} {flag.description}")


# Setup feature flags for TechMart
ff = FeatureFlagManager()

ff.register("new_customer_scoring",
    enabled=True, rollout_pct=25,
    description="ML-based customer scoring model",
    owner="data-science")

ff.register("realtime_inventory",
    enabled=False, rollout_pct=100,
    description="Real-time inventory updates (streaming)",
    owner="platform-team")

ff.register("enhanced_fraud_detection",
    enabled=True, rollout_pct=100,
    description="New fraud detection rules v2",
    owner="risk-team")

ff.register("deprecated_legacy_etl",
    enabled=True, rollout_pct=50,
    description="Gradual migration from legacy ETL",
    owner="migration-team",
    expires=date(2025, 6, 30))

print("🚩 Feature Flags Status:")
ff.status()


In [0]:
# Using feature flags in a pipeline
from pyspark.sql.functions import col, when, lit, udf
from pyspark.sql.types import StringType

def run_pipeline_with_flags(ff_manager):
    """Run order processing pipeline with feature flags."""
    print("🚀 Running Order Processing Pipeline with Feature Flags\n")
    
    df = spark.table("techmart_dw.orders").limit(100)
    print(f"Input: {df.count()} orders")
    
    # Feature: New scoring model (25% rollout)
    if ff_manager.is_enabled("new_customer_scoring"):
        print("\n📊 new_customer_scoring: ENABLED (25% rollout)")
        # Apply new scoring to subset of customers
        scored_count = 0
        not_scored_count = 0
        rows = df.select("customer_id").distinct().collect()
        for row in rows:
            if ff_manager.is_enabled("new_customer_scoring", entity_id=row.customer_id):
                scored_count += 1
            else:
                not_scored_count += 1
        print(f"   New model applied to: {scored_count} customers")
        print(f"   Legacy model for: {not_scored_count} customers")
    else:
        print("\n📊 new_customer_scoring: DISABLED — using legacy model")
    
    # Feature: Enhanced fraud detection (100% rollout)
    if ff_manager.is_enabled("enhanced_fraud_detection"):
        print("\n🔒 enhanced_fraud_detection: ENABLED (100%)")
        # Apply enhanced rules
        high_risk = df.filter(col("total_amount") > 8000).count()
        print(f"   Flagged {high_risk} high-risk orders for review")
    else:
        print("\n🔒 enhanced_fraud_detection: DISABLED")
    
    # Feature: Realtime inventory (disabled)
    if ff_manager.is_enabled("realtime_inventory"):
        print("\n📦 realtime_inventory: ENABLED — using streaming updates")
    else:
        print("\n📦 realtime_inventory: DISABLED — using batch updates (default)")
    
    print("\n✅ Pipeline complete")

run_pipeline_with_flags(ff)

# Demonstrate kill switch
print("\n" + "="*60)
print("⚠️  INCIDENT: Fraud detection causing false positives!")
print("   Executing kill switch...")
ff.disable("enhanced_fraud_detection")
print("   ✅ enhanced_fraud_detection DISABLED")
print("\n🔄 Re-running pipeline after kill switch:")
run_pipeline_with_flags(ff)


In [0]:
# ============================================
# 🎯 YOUR TURN: Implement Feature Flags
# ============================================
# Create a feature flag system for a new "customer_360" pipeline:
#
# Flags to register:
# 1. "use_ml_segmentation" — enabled, 50% rollout, owner: "ml-team"
# 2. "include_social_data" — disabled, description: "Merge social media signals"
# 3. "new_ltv_formula" — enabled, 10% rollout, owner: "analytics"
#
# Then write a function that:
# - Reads 50 customers
# - If use_ml_segmentation is enabled for a customer_id, print "ML segment"
# - If new_ltv_formula is enabled for a customer_id, print "New LTV"
# - Count how many customers get each feature
#
# Write your code below:


In [0]:
# ============================================
# ✅ SOLUTION
# ============================================
ff2 = FeatureFlagManager()
ff2.register("use_ml_segmentation", enabled=True, rollout_pct=50, owner="ml-team",
             description="ML-based customer segmentation")
ff2.register("include_social_data", enabled=False, 
             description="Merge social media signals", owner="data-team")
ff2.register("new_ltv_formula", enabled=True, rollout_pct=10, owner="analytics",
             description="Updated LTV calculation formula")

print("🚩 Customer 360 Feature Flags:")
ff2.status()

# Process customers with flags
customers = spark.sql("SELECT customer_id FROM techmart_dw.customers LIMIT 50").collect()

ml_count = 0
ltv_count = 0
both_count = 0

for cust in customers:
    cid = cust.customer_id
    has_ml = ff2.is_enabled("use_ml_segmentation", entity_id=cid)
    has_ltv = ff2.is_enabled("new_ltv_formula", entity_id=cid)
    
    if has_ml:
        ml_count += 1
    if has_ltv:
        ltv_count += 1
    if has_ml and has_ltv:
        both_count += 1

print(f"\n📊 Feature Distribution (50 customers):")
print(f"  ML Segmentation (50% target): {ml_count} customers ({ml_count*2}%)")
print(f"  New LTV Formula (10% target): {ltv_count} customers ({ltv_count*2}%)")
print(f"  Both features: {both_count} customers")
print(f"  Neither: {50 - ml_count - ltv_count + both_count} customers")


---
# 📗 Section 10: Databricks-Specific Deployment Patterns

## Declarative Automation Bundles (formerly Databricks Asset Bundles)

Declarative Automation Bundles (DABs) are the recommended way to deploy Databricks projects.
They define your entire project — pipelines, jobs, notebooks — as code.

**Reference:** https://docs.databricks.com/aws/dev-tools/bundles/

### Bundle Structure
```
my-project/
├── databricks.yml          # Main bundle config
├── src/
│   ├── pipelines/
│   │   └── orders_pipeline.py
│   └── notebooks/
│       └── analysis.py
├── resources/
│   ├── jobs.yml            # Job definitions
│   └── pipelines.yml       # Pipeline definitions
├── tests/
│   └── test_transforms.py
└── fixtures/
    └── test_data.json
```

### Key CLI Commands
```bash
databricks bundle validate    # Check config is valid
databricks bundle deploy      # Deploy to target environment
databricks bundle run         # Execute a job/pipeline
databricks bundle destroy     # Remove deployed resources
```

### Targets (Environments)
```yaml
# databricks.yml
targets:
  dev:
    default: true
    workspace:
      host: https://dev.cloud.databricks.com
  staging:
    workspace:
      host: https://staging.cloud.databricks.com
  prod:
    workspace:
      host: https://prod.cloud.databricks.com
    run_as:
      service_principal_name: prod-deployer
```

> 📘 See Notebook #35 for comprehensive DABs coverage including variables, 
> permissions, and CI/CD integration patterns.

In [0]:
# Simulate a databricks.yml bundle configuration
import json

databricks_bundle = {
    "bundle": {
        "name": "techmart-data-platform"
    },
    "variables": {
        "catalog": {
            "description": "Unity Catalog name",
            "default": "dev_catalog"
        },
        "schema": {
            "description": "Schema/database name",
            "default": "techmart_dw"
        },
        "warehouse_id": {
            "description": "SQL Warehouse ID for queries"
        }
    },
    "targets": {
        "dev": {
            "default": True,
            "workspace": {
                "host": "https://dbc-dev-12345.cloud.databricks.com"
            },
            "variables": {
                "catalog": "dev_catalog",
                "schema": "dev_techmart"
            }
        },
        "staging": {
            "workspace": {
                "host": "https://dbc-staging-12345.cloud.databricks.com"
            },
            "variables": {
                "catalog": "staging_catalog",
                "schema": "staging_techmart"
            }
        },
        "prod": {
            "workspace": {
                "host": "https://dbc-prod-12345.cloud.databricks.com"
            },
            "variables": {
                "catalog": "prod_catalog",
                "schema": "prod_techmart"
            },
            "run_as": {
                "service_principal_name": "prod-pipeline-deployer"
            }
        }
    },
    "resources": {
        "jobs": {
            "daily_orders_pipeline": {
                "name": "Daily Orders Pipeline",
                "schedule": {
                    "quartz_cron_expression": "0 0 6 * * ?",
                    "timezone_id": "America/New_York"
                },
                "tasks": [
                    {
                        "task_key": "ingest_orders",
                        "notebook_task": {
                            "notebook_path": "src/pipelines/bronze/ingest_orders.py"
                        }
                    },
                    {
                        "task_key": "clean_orders",
                        "depends_on": [{"task_key": "ingest_orders"}],
                        "notebook_task": {
                            "notebook_path": "src/pipelines/silver/clean_orders.py"
                        }
                    },
                    {
                        "task_key": "aggregate_sales",
                        "depends_on": [{"task_key": "clean_orders"}],
                        "notebook_task": {
                            "notebook_path": "src/pipelines/gold/daily_sales.py"
                        }
                    }
                ]
            }
        }
    }
}

print("📄 databricks.yml (Declarative Automation Bundle config):")
print("=" * 60)

# Print as YAML-like format for readability
def print_yaml(obj, indent=0):
    """Print dict as YAML-like format."""
    prefix = "  " * indent
    if isinstance(obj, dict):
        for key, value in obj.items():
            if isinstance(value, (dict, list)):
                print(f"{prefix}{key}:")
                print_yaml(value, indent + 1)
            else:
                print(f"{prefix}{key}: {value}")
    elif isinstance(obj, list):
        for item in obj:
            if isinstance(item, dict):
                print(f"{prefix}-")
                print_yaml(item, indent + 1)
            else:
                print(f"{prefix}- {item}")

print_yaml(databricks_bundle)


In [0]:
# Simulate bundle deployment workflow
class BundleDeployer:
    """Simulates Databricks bundle deployment commands."""
    
    def __init__(self, bundle_config):
        self.config = bundle_config
        self.deployed_targets = {}
    
    def validate(self, target="dev"):
        """Simulate: databricks bundle validate -t <target>"""
        print(f"$ databricks bundle validate -t {target}")
        print(f"  Validating bundle '{self.config['bundle']['name']}'...")
        
        errors = []
        # Check required fields
        if "targets" not in self.config:
            errors.append("Missing 'targets' section")
        if target not in self.config.get("targets", {}):
            errors.append(f"Target '{target}' not defined")
        if "resources" not in self.config:
            errors.append("Missing 'resources' section")
        
        if errors:
            print(f"  ❌ Validation FAILED:")
            for e in errors:
                print(f"     - {e}")
            return False
        
        # Check variables are resolved
        target_vars = self.config["targets"][target].get("variables", {})
        print(f"  ✅ Target '{target}' is valid")
        print(f"     Workspace: {self.config['targets'][target]['workspace']['host']}")
        print(f"     Variables: {target_vars}")
        return True
    
    def deploy(self, target="dev"):
        """Simulate: databricks bundle deploy -t <target>"""
        print(f"\n$ databricks bundle deploy -t {target}")
        
        if not self.validate(target):
            return False
        
        print(f"  Deploying to {target}...")
        resources = self.config.get("resources", {})
        
        for resource_type, items in resources.items():
            for name, config in items.items():
                print(f"  📦 Deploying {resource_type}/{name}...")
                print(f"     Name: {config.get('name', name)}")
                if "tasks" in config:
                    print(f"     Tasks: {len(config['tasks'])}")
        
        self.deployed_targets[target] = {
            "deployed_at": datetime.now().isoformat(),
            "resources": list(resources.keys())
        }
        print(f"  ✅ Deployment to {target} complete!")
        return True
    
    def destroy(self, target="dev"):
        """Simulate: databricks bundle destroy -t <target>"""
        print(f"\n$ databricks bundle destroy -t {target}")
        if target in self.deployed_targets:
            print(f"  🗑️ Removing all resources from {target}...")
            del self.deployed_targets[target]
            print(f"  ✅ All resources removed from {target}")
        else:
            print(f"  ⚠️ Nothing deployed to {target}")


# Demo the deployment workflow
deployer = BundleDeployer(databricks_bundle)

print("=" * 60)
print("DEPLOYMENT WORKFLOW: dev → staging → prod")
print("=" * 60)

# Step 1: Deploy to dev
deployer.deploy("dev")

# Step 2: Deploy to staging
deployer.deploy("staging")

# Step 3: Deploy to prod
deployer.deploy("prod")

print(f"\n📋 Deployment Summary:")
for target, info in deployer.deployed_targets.items():
    print(f"  {target}: deployed at {info['deployed_at'][:19]}")


In [0]:
# ============================================
# 🎯 YOUR TURN: Design a DABs Deployment
# ============================================
# Create a databricks_bundle config (as a Python dict) for:
# - Bundle name: "customer-analytics-platform"
# - 3 targets: dev, staging, prod (with different catalogs)
# - Variables: catalog, schema, notification_email
# - One job: "weekly_customer_segmentation" with 3 tasks:
#   1. extract_customer_data
#   2. run_segmentation_model (depends on task 1)
#   3. update_dashboards (depends on task 2)
#
# Then use BundleDeployer to validate and deploy to staging.
#
# Write your code below:


In [0]:
# ============================================
# ✅ SOLUTION
# ============================================
customer_bundle = {
    "bundle": {
        "name": "customer-analytics-platform"
    },
    "variables": {
        "catalog": {"description": "Unity Catalog", "default": "dev_catalog"},
        "schema": {"description": "Target schema", "default": "analytics"},
        "notification_email": {"description": "Alert recipient"}
    },
    "targets": {
        "dev": {
            "default": True,
            "workspace": {"host": "https://dev.databricks.com"},
            "variables": {"catalog": "dev_catalog", "schema": "dev_analytics",
                         "notification_email": "dev@company.com"}
        },
        "staging": {
            "workspace": {"host": "https://staging.databricks.com"},
            "variables": {"catalog": "staging_catalog", "schema": "staging_analytics",
                         "notification_email": "qa@company.com"}
        },
        "prod": {
            "workspace": {"host": "https://prod.databricks.com"},
            "variables": {"catalog": "prod_catalog", "schema": "prod_analytics",
                         "notification_email": "oncall@company.com"},
            "run_as": {"service_principal_name": "analytics-deployer"}
        }
    },
    "resources": {
        "jobs": {
            "weekly_customer_segmentation": {
                "name": "Weekly Customer Segmentation",
                "schedule": {"quartz_cron_expression": "0 0 2 ? * MON", "timezone_id": "UTC"},
                "tasks": [
                    {
                        "task_key": "extract_customer_data",
                        "notebook_task": {"notebook_path": "src/extract/customers.py"}
                    },
                    {
                        "task_key": "run_segmentation_model",
                        "depends_on": [{"task_key": "extract_customer_data"}],
                        "notebook_task": {"notebook_path": "src/models/segmentation.py"}
                    },
                    {
                        "task_key": "update_dashboards",
                        "depends_on": [{"task_key": "run_segmentation_model"}],
                        "notebook_task": {"notebook_path": "src/reporting/refresh_dashboards.py"}
                    }
                ]
            }
        }
    }
}

# Deploy
deployer2 = BundleDeployer(customer_bundle)
deployer2.deploy("staging")


---
# 📗 Section 11: Mini Projects

## Project 1: Pipeline as a Package

Refactor a pipeline into a proper Python package structure with:
- Modular source code
- Test suite
- Environment configs
- Build/deploy tooling

In [0]:
# PROJECT 1: Pipeline as a Package
# Implement a complete pipeline package structure in Python

class PipelinePackage:
    """Represents a deployable pipeline package."""
    
    def __init__(self, name, version):
        self.name = name
        self.version = version
        self.modules = {}
        self.tests = {}
        self.configs = {}
    
    def add_module(self, path, code):
        self.modules[path] = code
    
    def add_test(self, path, code):
        self.tests[path] = code
    
    def add_config(self, env, config):
        self.configs[env] = config
    
    def show_structure(self):
        print(f"📦 {self.name} v{self.version}")
        print(f"├── src/")
        for path in sorted(self.modules.keys()):
            print(f"│   └── {path}")
        print(f"├── tests/")
        for path in sorted(self.tests.keys()):
            print(f"│   └── {path}")
        print(f"├── config/")
        for env in sorted(self.configs.keys()):
            print(f"│   └── {env}.yaml")
        print(f"├── pyproject.toml")
        print(f"└── README.md")


# Build the TechMart orders pipeline package
pkg = PipelinePackage("techmart-orders-pipeline", "1.0.0")

# Source modules
pkg.add_module("pipelines/orders/__init__.py", "")
pkg.add_module("pipelines/orders/extract.py", """
from pyspark.sql import DataFrame

def extract_orders(spark, config) -> DataFrame:
    """Extract orders from source table."""
    source = config['source']
    df = spark.table(f"{source['catalog']}.{source['schema']}.{source['table']}")
    if config.get('sample_fraction', 1.0) < 1.0:
        df = df.sample(fraction=config['sample_fraction'])
    return df
""")

pkg.add_module("pipelines/orders/transform.py", """
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, when, current_timestamp

def clean_orders(df: DataFrame) -> DataFrame:
    """Remove invalid orders and standardize."""
    return df.filter(col('status').isNotNull()).filter(col('total_amount') > 0)

def add_tiers(df: DataFrame, thresholds: dict) -> DataFrame:
    """Add amount tier classification."""
    return df.withColumn('amount_tier',
        when(col('total_amount') > thresholds['high'], 'high')
        .when(col('total_amount') > thresholds['medium'], 'medium')
        .otherwise('low'))

def add_audit_columns(df: DataFrame) -> DataFrame:
    """Add standard audit columns."""
    return df.withColumn('_processed_at', current_timestamp())
""")

pkg.add_module("pipelines/orders/load.py", """
from pyspark.sql import DataFrame

def write_to_table(df: DataFrame, config: dict) -> None:
    """Write DataFrame to target table."""
    target = config['target']
    table_name = f"{target['catalog']}.{target['schema']}.{target['table']}"
    df.write.mode(target.get('mode', 'append')).saveAsTable(table_name)
""")

pkg.add_module("pipelines/orders/pipeline.py", """
from .extract import extract_orders
from .transform import clean_orders, add_tiers, add_audit_columns
from .load import write_to_table

def run(spark, config):
    """Execute the full orders pipeline."""
    # Extract
    raw = extract_orders(spark, config)
    
    # Transform
    cleaned = clean_orders(raw)
    tiered = add_tiers(cleaned, config['thresholds'])
    final = add_audit_columns(tiered)
    
    # Load
    write_to_table(final, config)
    return final
""")

# Tests
pkg.add_test("test_transform.py", """
def test_clean_orders_removes_nulls(spark, sample_orders):
    from pipelines.orders.transform import clean_orders
    result = clean_orders(sample_orders)
    assert result.filter('status IS NULL').count() == 0

def test_add_tiers_assigns_correctly(spark, sample_orders):
    from pipelines.orders.transform import add_tiers
    thresholds = {'high': 5000, 'medium': 1000}
    result = add_tiers(sample_orders, thresholds)
    assert 'amount_tier' in result.columns
""")

pkg.add_test("test_pipeline_e2e.py", """
def test_full_pipeline_produces_output(spark, config):
    from pipelines.orders.pipeline import run
    result = run(spark, config)
    assert result.count() > 0
    assert '_processed_at' in result.columns
""")

# Configs
pkg.add_config("dev", {
    "source": {"catalog": "dev_catalog", "schema": "techmart", "table": "orders"},
    "target": {"catalog": "dev_catalog", "schema": "techmart", "table": "gold_orders", "mode": "overwrite"},
    "thresholds": {"high": 5000, "medium": 1000},
    "sample_fraction": 0.01
})

pkg.add_config("prod", {
    "source": {"catalog": "prod_catalog", "schema": "techmart", "table": "orders"},
    "target": {"catalog": "prod_catalog", "schema": "techmart", "table": "gold_orders", "mode": "append"},
    "thresholds": {"high": 5000, "medium": 1000},
    "sample_fraction": 1.0
})

# Show the package
pkg.show_structure()

# Show pyproject.toml
print("\n📄 pyproject.toml:")
pyproject = """
[project]
name = "techmart-orders-pipeline"
version = "1.0.0"
description = "TechMart order processing pipeline"
requires-python = ">=3.9"
dependencies = [
    "pyspark>=3.4.0",
    "delta-spark>=2.4.0",
    "pyyaml>=6.0",
]

[project.optional-dependencies]
test = ["pytest>=7.0", "pytest-cov>=4.0", "chispa>=0.9"]
dev = ["ruff>=0.1.0", "black>=23.0", "mypy>=1.0"]

[tool.pytest.ini_options]
testpaths = ["tests"]
addopts = "-v --cov=src"

[tool.ruff]
line-length = 120
target-version = "py39"
"""
print(pyproject)


## Project 2: Deployment Simulator

Build a complete deployment system that validates, deploys, tests, and handles rollback.

In [0]:
# PROJECT 2: Full Deployment Simulator
from datetime import datetime
from enum import Enum

class Environment(Enum):
    DEV = "dev"
    STAGING = "staging"
    PROD = "prod"


class DeploymentPlan:
    """A deployment plan with validation, execution, and rollback."""
    
    def __init__(self, pipeline_name, version, target_env):
        self.pipeline_name = pipeline_name
        self.version = version
        self.target_env = target_env
        self.steps = []
        self.executed_steps = []
        self.status = "planned"
        self.started_at = None
        self.completed_at = None
    
    def add_step(self, name, action, rollback_action=None, critical=True):
        """Add a deployment step."""
        self.steps.append({
            "name": name,
            "action": action,
            "rollback_action": rollback_action,
            "critical": critical,
            "status": "pending"
        })
    
    def execute(self):
        """Execute the deployment plan."""
        self.started_at = datetime.now()
        self.status = "executing"
        
        print(f"{'='*70}")
        print(f"🚀 DEPLOYMENT PLAN: {self.pipeline_name} {self.version}")
        print(f"   Target: {self.target_env.value}")
        print(f"   Steps: {len(self.steps)}")
        print(f"   Started: {self.started_at.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"{'='*70}")
        
        for i, step in enumerate(self.steps, 1):
            print(f"
  Step {i}/{len(self.steps)}: {step['name']}")
            try:
                result = step["action"]()
                step["status"] = "completed"
                self.executed_steps.append(step)
                print(f"  ✅ {step['name']}: {result}")
            except Exception as e:
                step["status"] = "failed"
                print(f"  ❌ {step['name']}: FAILED — {e}")
                
                if step["critical"]:
                    print(f"
  🛑 Critical step failed! Initiating rollback...")
                    self._rollback()
                    self.status = "rolled_back"
                    return False
                else:
                    print(f"  ⚠️ Non-critical step failed, continuing...")
        
        self.status = "completed"
        self.completed_at = datetime.now()
        duration = (self.completed_at - self.started_at).total_seconds()
        print(f"
{'='*70}")
        print(f"✅ DEPLOYMENT COMPLETE in {duration:.1f}s")
        print(f"{'='*70}")
        return True
    
    def _rollback(self):
        """Rollback executed steps in reverse order."""
        print(f"
  {'─'*50}")
        print(f"  ⏪ ROLLBACK: Reverting {len(self.executed_steps)} steps")
        print(f"  {'─'*50}")
        
        for step in reversed(self.executed_steps):
            if step["rollback_action"]:
                try:
                    step["rollback_action"]()
                    print(f"  ↩️ Rolled back: {step['name']}")
                except Exception as e:
                    print(f"  ⚠️ Rollback failed for {step['name']}: {e}")
            else:
                print(f"  ⏭️ No rollback for: {step['name']}")
    
    def summary(self):
        """Print deployment summary."""
        print(f"
📋 Deployment Summary:")
        print(f"   Pipeline: {self.pipeline_name}")
        print(f"   Version: {self.version}")
        print(f"   Target: {self.target_env.value}")
        print(f"   Status: {self.status}")
        print(f"   Steps:")
        for step in self.steps:
            icon = {"completed": "✅", "failed": "❌", "pending": "⏳"}[step["status"]]
            print(f"     {icon} {step['name']}")


# Build a deployment plan for the orders pipeline
plan = DeploymentPlan("techmart-orders-pipeline", "v1.3.0", Environment.STAGING)

# Step 1: Validate config
plan.add_step(
    "Validate configuration",
    action=lambda: "Config valid for staging environment",
    rollback_action=None
)

# Step 2: Run pre-deployment tests
plan.add_step(
    "Pre-deployment tests",
    action=lambda: f"12 tests passed ({spark.table('techmart_dw.orders').count()} rows accessible)",
    rollback_action=None
)

# Step 3: Create backup
backup_created = [False]
def create_backup():
    backup_created[0] = True
    return "Backup created: techmart_dw.orders_backup_20240601"

def remove_backup():
    backup_created[0] = False

plan.add_step(
    "Create table backup",
    action=create_backup,
    rollback_action=remove_backup
)

# Step 4: Apply schema migration
plan.add_step(
    "Apply schema migration",
    action=lambda: "Added column: amount_tier STRING",
    rollback_action=lambda: None  # Would ALTER TABLE DROP COLUMN
)

# Step 5: Deploy new pipeline code
plan.add_step(
    "Deploy pipeline code",
    action=lambda: "Pipeline v1.3.0 deployed to staging workspace",
    rollback_action=lambda: None  # Would restore previous version
)

# Step 6: Run smoke tests
plan.add_step(
    "Smoke tests",
    action=lambda: f"3/3 smoke tests passed",
    rollback_action=None
)

# Step 7: Update monitoring (non-critical)
plan.add_step(
    "Update monitoring dashboards",
    action=lambda: "Dashboards updated with new metrics",
    rollback_action=None,
    critical=False
)

# Execute!
plan.execute()
plan.summary()


In [0]:
# Demo: Deployment with failure and rollback
print("\n" + "="*70)
print("SCENARIO: Deployment that fails and triggers rollback")
print("="*70)

plan_fail = DeploymentPlan("customer-360-pipeline", "v2.0.0", Environment.PROD)

steps_executed = []

plan_fail.add_step(
    "Validate prod config",
    action=lambda: (steps_executed.append("validate"), "Config valid")[1],
    rollback_action=None
)

plan_fail.add_step(
    "Create prod backup",
    action=lambda: (steps_executed.append("backup"), "Backup: customer_360_backup_v1.9")[1],
    rollback_action=lambda: steps_executed.append("remove_backup")
)

plan_fail.add_step(
    "Deploy new code",
    action=lambda: (steps_executed.append("deploy"), "Code deployed")[1],
    rollback_action=lambda: steps_executed.append("restore_old_code")
)

def failing_smoke_test():
    """This smoke test will fail."""
    raise AssertionError("Data quality check failed: 23% NULL rate in customer_segment (threshold: 5%)")

plan_fail.add_step(
    "Smoke tests",
    action=failing_smoke_test,
    rollback_action=None,
    critical=True  # This will trigger rollback
)

plan_fail.add_step(
    "Update dashboards",
    action=lambda: "Updated",
    rollback_action=None
)

plan_fail.execute()
plan_fail.summary()


---
# 📗 Section 12: Interview Questions

## Common CI/CD & Version Control Interview Questions for Data Engineers

These questions test your understanding of production-grade pipeline management.

In [0]:
# Interview Question 1: Environment Management
print("""
╔══════════════════════════════════════════════════════════════════════╗
║  INTERVIEW Q1: How do you manage multiple environments for         ║
║  data pipelines?                                                    ║
╚══════════════════════════════════════════════════════════════════════╝

STRONG ANSWER:

"We use a configuration-driven approach with three environments:
dev, staging, and production.

1. CONFIGURATION SEPARATION:
   - Each environment has its own config file (dev.yaml, prod.yaml)
   - Configs specify: database names, storage paths, batch sizes,
     quality thresholds, and alert recipients
   - Code reads config at runtime — same code, different behavior

2. NAMING CONVENTIONS:
   - Databases: dev_techmart, staging_techmart, prod_techmart
   - Storage: /Volumes/dev/data/, /Volumes/prod/data/
   - Jobs: [dev] Daily Sales, [prod] Daily Sales

3. ACCESS CONTROL:
   - Dev: all engineers have read/write
   - Staging: engineers read, CI/CD writes
   - Prod: only service principals can write

4. DEPLOYMENT FLOW:
   - Code merges to main → auto-deploy to staging
   - Smoke tests pass in staging → manual approval for prod
   - Rollback: revert to previous version via Git tag

5. TOOLS:
   - Declarative Automation Bundles (DABs) with targets for each env
   - Unity Catalog for data governance across environments
   - Terraform/Pulumi for infrastructure"
""")


In [0]:
# Interview Question 2: CI/CD Strategy
print("""
╔══════════════════════════════════════════════════════════════════════╗
║  INTERVIEW Q2: Describe your CI/CD strategy for data engineering   ║
╚══════════════════════════════════════════════════════════════════════╝

STRONG ANSWER:

"Our CI/CD pipeline has distinct stages optimized for data workloads:

CI (on every PR):
  1. LINT: ruff + black for code style (< 30 seconds)
  2. UNIT TESTS: pytest on transformation functions (< 2 minutes)
     - Test with small DataFrames, no cluster needed
  3. CONTRACT TESTS: validate schemas match expectations
  4. INTEGRATION TESTS: run pipeline on sample data (< 5 minutes)
     - Use a dedicated test cluster or serverless
  5. DATA QUALITY: run expectations on test output

CD (on merge to main):
  1. BUILD: package code as wheel/egg
  2. DEPLOY TO STAGING: via DABs CLI
  3. SMOKE TESTS: verify pipeline runs end-to-end in staging
  4. APPROVAL GATE: manual review for prod
  5. DEPLOY TO PROD: via DABs with service principal
  6. MONITOR: check first run succeeds, alert on failure

KEY DECISIONS:
  - We use trunk-based development (short-lived branches)
  - Feature flags for incomplete features
  - Canary deployments for high-risk changes
  - Automated rollback if smoke tests fail"
""")


In [0]:
# Interview Question 3: Schema Migrations
print("""
╔══════════════════════════════════════════════════════════════════════╗
║  INTERVIEW Q3: How do you handle schema migrations in production?  ║
╚══════════════════════════════════════════════════════════════════════╝

STRONG ANSWER:

"Schema migrations are treated as code, versioned in Git:

1. MIGRATION FILES:
   - Numbered sequentially: 001_create_table.sql, 002_add_column.sql
   - Each has UP (apply) and DOWN (rollback) SQL
   - Tracked in a schema_migrations table

2. SAFE MIGRATION PATTERNS:
   - ADD column: always safe (backward compatible)
   - RENAME column: create new, copy data, drop old (3 migrations)
   - DROP column: deprecate first, remove after downstream updated
   - TYPE change: add new column, migrate data, swap

3. BREAKING CHANGES:
   - Never break downstream consumers
   - Use expand-and-contract pattern:
     Step 1: Add new column (expand)
     Step 2: Backfill data
     Step 3: Update consumers to use new column
     Step 4: Remove old column (contract)

4. DELTA LAKE ADVANTAGES:
   - Schema evolution with mergeSchema option
   - Time travel for rollback (RESTORE TABLE AS OF VERSION)
   - Column mapping for renames without rewriting data

5. TESTING:
   - Run migrations in staging first
   - Verify downstream queries still work
   - Check data quality after migration"
""")


In [0]:
# Interview Question 4: Branching Strategy
print("""
╔══════════════════════════════════════════════════════════════════════╗
║  INTERVIEW Q4: What's your branching strategy for DE projects?     ║
╚══════════════════════════════════════════════════════════════════════╝

STRONG ANSWER:

"We use trunk-based development with these conventions:

BRANCH NAMING:
  - feature/JIRA-123-add-customer-tier
  - fix/JIRA-456-null-handling-orders
  - hotfix/JIRA-789-prod-pipeline-failure

WORKFLOW:
  1. Create branch from main
  2. Make changes (small, focused PRs)
  3. Open PR → triggers CI pipeline
  4. Code review (at least 1 approval)
  5. Merge to main → auto-deploy to staging
  6. Promote to prod after staging validation

RULES:
  - Branches live < 3 days (avoid merge conflicts)
  - No direct commits to main
  - Squash merge for clean history
  - Feature flags for work-in-progress
  - Hotfix branches can bypass staging (with approval)

WHY NOT GITFLOW:
  - Too complex for most DE teams
  - Long-lived branches cause painful merges
  - Data pipelines benefit from frequent, small deploys
  - Exception: regulated industries may need release branches"
""")


In [0]:
# Interview Question 5: Rollback Strategy
print("""
╔══════════════════════════════════════════════════════════════════════╗
║  INTERVIEW Q5: How do you roll back a failed pipeline deployment?  ║
╚══════════════════════════════════════════════════════════════════════╝

STRONG ANSWER:

"Our rollback strategy depends on what failed:

1. CODE ROLLBACK (most common):
   - Revert the Git commit (git revert)
   - Redeploy previous version via DABs
   - Takes < 5 minutes

2. DATA ROLLBACK:
   - Delta Lake time travel: RESTORE TABLE AS OF VERSION X
   - Or: RESTORE TABLE AS OF TIMESTAMP '2024-01-15 10:00:00'
   - Instant for recent versions (within retention period)

3. SCHEMA ROLLBACK:
   - Run the DOWN migration script
   - Or use Delta column mapping to undo renames
   - More complex — may need data backfill

4. PREVENTION (better than cure):
   - Blue-green deployment: keep old version running
   - Canary: process 5% of data with new code first
   - Shadow mode: run new code alongside, compare outputs
   - Automated smoke tests catch issues before full rollout

5. INCIDENT RESPONSE:
   - Automated alerts on pipeline failure
   - Runbook with step-by-step rollback instructions
   - Post-mortem after every production incident
   - Blameless culture — focus on process improvement"
""")


In [0]:
# Interview Question 6: Testing Data Pipelines
print("""
╔══════════════════════════════════════════════════════════════════════╗
║  INTERVIEW Q6: How do you test data pipelines before production?   ║
╚══════════════════════════════════════════════════════════════════════╝

STRONG ANSWER:

"We have a multi-layer testing strategy:

1. UNIT TESTS (every commit, < 2 min):
   - Test individual transformation functions
   - Use small, in-memory DataFrames
   - Framework: pytest + chispa (for Spark DataFrame assertions)
   - Example: test that clean_email() handles NULLs correctly

2. INTEGRATION TESTS (every PR, < 10 min):
   - Test pipeline end-to-end with sample data
   - Verify correct table writes, schema, row counts
   - Use a dedicated test database (auto-cleaned)

3. CONTRACT TESTS (every PR, < 1 min):
   - Verify input/output schemas match expectations
   - Catch breaking changes from upstream teams
   - Schema registry or JSON schema files in Git

4. DATA QUALITY TESTS (every pipeline run):
   - NULL rate checks (< 1% for critical columns)
   - Uniqueness constraints (no duplicate PKs)
   - Referential integrity (all FKs exist in parent)
   - Freshness checks (data not stale)
   - Volume checks (row count within expected range)

5. PERFORMANCE TESTS (before major releases):
   - Run with production-scale data in staging
   - Verify SLAs are met (pipeline completes in < X minutes)
   - Check resource usage (cluster cost)

6. CHAOS TESTS (quarterly):
   - Simulate upstream failures
   - Test retry logic and error handling
   - Verify alerting works correctly"
""")


---
# 🧹 Cleanup

Remove tables created during this notebook.

In [0]:
# Cleanup tables created in this notebook
cleanup_tables = [
    "techmart_dw.schema_migrations",
    "techmart_dw.gold_daily_metrics",
    "techmart_dw.ltv_history"
]

cleanup_views = [
    "techmart_dw.v_ltv_changes"
]

for table in cleanup_tables:
    try:
        spark.sql(f"DROP TABLE IF EXISTS {table}")
        print(f"  Dropped table: {table}")
    except Exception as e:
        print(f"  Skip {table}: {e}")

for view in cleanup_views:
    try:
        spark.sql(f"DROP VIEW IF EXISTS {view}")
        print(f"  Dropped view: {view}")
    except Exception as e:
        print(f"  Skip {view}: {e}")

# Try to drop added columns (may fail if not applied)
try:
    spark.sql("ALTER TABLE techmart_dw.customers DROP COLUMN IF EXISTS loyalty_tier")
    print("  Dropped column: customers.loyalty_tier")
except:
    pass

try:
    spark.sql("ALTER TABLE techmart_dw.customers DROP COLUMN IF EXISTS ltv_recalculated_at")
    print("  Dropped column: customers.ltv_recalculated_at")
except:
    pass

print("\n✅ Cleanup complete!")

## Testing Data Pipelines -- The Testing Pyramid

```
                    ┌─────────────────┐
                    │  End-to-End     │  ← Full pipeline run (slow, expensive)
                    │  Tests          │     Run: weekly or on release
                    ├─────────────────┤
                    │  Integration    │  ← Test with real data (medium)
                    │  Tests          │     Run: on every PR
                    ├─────────────────┤
                    │  Unit Tests     │  ← Test individual functions (fast, cheap)
                    │                 │     Run: on every commit
                    └─────────────────┘
```

### Unit Tests for Data Transformations

```python
# tests/test_transforms.py
import pytest
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DecimalType
from src.transforms import clean_orders, calculate_revenue

@pytest.fixture(scope="session")
def spark():
    return SparkSession.builder.master("local[2]").appName("tests").getOrCreate()

def test_clean_orders_removes_nulls(spark):
    # Arrange
    schema = StructType([
        StructField("order_id", IntegerType()),
        StructField("amount", DecimalType(12,2)),
        StructField("status", StringType()),
    ])
    input_data = [(1, 100.0, "completed"), (None, 50.0, "pending"), (3, 200.0, "shipped")]
    df = spark.createDataFrame(input_data, schema)
    
    # Act
    result = clean_orders(df)
    
    # Assert
    assert result.count() == 2  # NULL order_id removed
    assert result.filter("order_id IS NULL").count() == 0

def test_clean_orders_standardizes_status(spark):
    input_data = [(1, 100.0, "COMPLETED"), (2, 200.0, "  Pending  ")]
    df = spark.createDataFrame(input_data, ["order_id", "amount", "status"])
    result = clean_orders(df)
    statuses = [r.status for r in result.collect()]
    assert all(s == s.lower().strip() for s in statuses)

def test_calculate_revenue_correct(spark):
    input_data = [(1, 100.0, "completed"), (2, 200.0, "completed"), (3, 50.0, "cancelled")]
    df = spark.createDataFrame(input_data, ["order_id", "amount", "status"])
    result = calculate_revenue(df)
    assert result == 300.0  # Only completed orders
```

### Integration Tests

```python
# tests/test_integration.py
def test_bronze_to_silver_pipeline(spark):
    # Use a small sample of real data
    bronze = spark.table("test.bronze_orders").limit(1000)
    
    # Run the actual transformation
    silver = transform_bronze_to_silver(bronze)
    
    # Assert quality
    assert silver.filter("order_id IS NULL").count() == 0
    assert silver.filter("total_amount <= 0").count() == 0
    assert silver.count() <= bronze.count()  # Can only remove rows, not add
```

In [0]:
# Unit testing patterns for data transformations
import unittest

# Simulate the transformation functions we want to test
def clean_orders(records):
    """Clean and validate order records."""
    cleaned = []
    for r in records:
        # Remove nulls
        if r.get("order_id") is None:
            continue
        if r.get("amount") is None:
            continue
        # Standardize
        cleaned.append({
            "order_id": r["order_id"],
            "amount": float(r["amount"]),
            "status": r.get("status", "").lower().strip(),
        })
    return cleaned

def calculate_revenue(records, status_filter="completed"):
    """Calculate total revenue for a given status."""
    return sum(r["amount"] for r in records if r["status"] == status_filter)

def deduplicate(records, key_field):
    """Remove duplicates, keeping the last occurrence."""
    seen = {}
    for r in records:
        seen[r[key_field]] = r
    return list(seen.values())


# Write unit tests
class TestDataTransformations(unittest.TestCase):
    
    def setUp(self):
        """Set up test data."""
        self.sample_orders = [
            {"order_id": 1, "amount": 100.0, "status": "COMPLETED"},
            {"order_id": 2, "amount": 200.0, "status": "  Pending  "},
            {"order_id": None, "amount": 50.0, "status": "shipped"},   # NULL id
            {"order_id": 4, "amount": None, "status": "completed"},    # NULL amount
            {"order_id": 5, "amount": 300.0, "status": "completed"},
        ]
    
    def test_clean_orders_removes_null_ids(self):
        result = clean_orders(self.sample_orders)
        self.assertEqual(len(result), 3)  # 2 nulls removed
        self.assertTrue(all(r["order_id"] is not None for r in result))
    
    def test_clean_orders_standardizes_status(self):
        result = clean_orders(self.sample_orders)
        for r in result:
            self.assertEqual(r["status"], r["status"].lower().strip())
    
    def test_calculate_revenue_completed_only(self):
        cleaned = clean_orders(self.sample_orders)
        revenue = calculate_revenue(cleaned, "completed")
        self.assertEqual(revenue, 400.0)  # 100 + 300
    
    def test_deduplicate_keeps_last(self):
        records = [
            {"id": 1, "value": "first"},
            {"id": 2, "value": "only"},
            {"id": 1, "value": "last"},  # Duplicate id=1
        ]
        result = deduplicate(records, "id")
        self.assertEqual(len(result), 2)
        id1_record = next(r for r in result if r["id"] == 1)
        self.assertEqual(id1_record["value"], "last")
    
    def test_empty_input(self):
        """Edge case: empty input should return empty output."""
        self.assertEqual(clean_orders([]), [])
        self.assertEqual(calculate_revenue([]), 0.0)
        self.assertEqual(deduplicate([], "id"), [])


# Run tests
print("🧪 RUNNING UNIT TESTS")
print("=" * 60)
loader = unittest.TestLoader()
suite = loader.loadTestsFromTestCase(TestDataTransformations)
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)

print(f"\n{'✅ ALL TESTS PASSED' if result.wasSuccessful() else '❌ SOME TESTS FAILED'}")
print(f"   Tests run: {result.testsRun}")
print(f"   Failures: {len(result.failures)}")
print(f"   Errors: {len(result.errors)}")


## GitHub Actions for Databricks -- Complete Workflow

### Full CI/CD Pipeline

```yaml
# .github/workflows/ci-cd.yml
name: Data Pipeline CI/CD

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

env:
  PYTHON_VERSION: "3.10"
  DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }}
  DATABRICKS_TOKEN: ${{ secrets.DATABRICKS_TOKEN }}

jobs:
  # ─── CI: Test every PR ───────────────────────────────────────
  test:
    name: Test
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      
      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: ${{ env.PYTHON_VERSION }}
      
      - name: Cache dependencies
        uses: actions/cache@v3
        with:
          path: ~/.cache/pip
          key: ${{ runner.os }}-pip-${{ hashFiles('requirements.txt') }}
      
      - name: Install dependencies
        run: |
          pip install -r requirements.txt
          pip install pytest pytest-cov ruff black mypy
      
      - name: Lint with ruff
        run: ruff check src/ tests/
      
      - name: Format check with black
        run: black --check src/ tests/
      
      - name: Type check with mypy
        run: mypy src/ --ignore-missing-imports
      
      - name: Run unit tests
        run: |
          pytest tests/unit/ -v --cov=src --cov-report=xml
      
      - name: Upload coverage
        uses: codecov/codecov-action@v3
        with:
          file: coverage.xml
      
      - name: Validate DABs bundle
        run: |
          pip install databricks-cli
          databricks bundle validate --target dev

  # ─── CD: Deploy on merge to main ─────────────────────────────
  deploy-dev:
    name: Deploy to DEV
    needs: test
    if: github.ref == 'refs/heads/main'
    runs-on: ubuntu-latest
    environment: dev
    steps:
      - uses: actions/checkout@v4
      
      - name: Deploy to DEV
        run: databricks bundle deploy --target dev
      
      - name: Run integration tests on DEV
        run: pytest tests/integration/ -v --target=dev
      
      - name: Notify Slack
        uses: slackapi/slack-github-action@v1
        with:
          payload: '{"text": "Deployed to DEV: ${{ github.sha }}"}'
        env:
          SLACK_WEBHOOK_URL: ${{ secrets.SLACK_WEBHOOK }}

  deploy-prod:
    name: Deploy to PROD
    needs: deploy-dev
    runs-on: ubuntu-latest
    environment: production  # Requires manual approval in GitHub
    steps:
      - uses: actions/checkout@v4
      
      - name: Deploy to PROD
        run: databricks bundle deploy --target prod
      
      - name: Run smoke tests
        run: pytest tests/smoke/ -v --target=prod
      
      - name: Create release tag
        run: |
          git tag "release-$(date +%Y%m%d-%H%M%S)"
          git push --tags
```

### Environment Secrets Setup

```
GitHub Repository Settings > Secrets and Variables > Actions:

DATABRICKS_HOST_DEV    = https://dev-workspace.cloud.databricks.com
DATABRICKS_TOKEN_DEV   = dapi_dev_token_here
DATABRICKS_HOST_PROD   = https://prod-workspace.cloud.databricks.com
DATABRICKS_TOKEN_PROD  = dapi_prod_token_here
SLACK_WEBHOOK          = https://hooks.slack.com/services/...
```

In [0]:
# CI/CD pipeline simulation
class CICDPipeline:
    """Simulates a complete CI/CD pipeline execution."""
    
    def __init__(self, commit_sha, branch):
        self.commit_sha = commit_sha[:7]
        self.branch = branch
        self.stages = []
        self.overall_status = "pending"
    
    def add_stage(self, name, steps):
        self.stages.append({"name": name, "steps": steps, "status": "pending"})
    
    def run(self):
        print(f"\n🚀 CI/CD PIPELINE")
        print(f"   Commit: {self.commit_sha} | Branch: {self.branch}")
        print("=" * 60)
        
        for stage in self.stages:
            print(f"\n--- {stage['name']} ---")
            stage_passed = True
            
            for step in stage["steps"]:
                passed = step.get("passes", True)
                duration = step.get("duration_sec", 5)
                status = "✅" if passed else "❌"
                print(f"   {status} {step['name']} ({duration}s)")
                
                if not passed:
                    stage_passed = False
                    print(f"      Error: {step.get('error', 'Step failed')}")
                    if step.get("blocking", True):
                        print(f"   ⛔ Stage '{stage['name']}' failed -- stopping pipeline")
                        self.overall_status = "failed"
                        return False
            
            stage["status"] = "passed" if stage_passed else "failed"
        
        self.overall_status = "passed"
        print(f"\n{'✅ PIPELINE PASSED' if self.overall_status == 'passed' else '❌ PIPELINE FAILED'}")
        return self.overall_status == "passed"


# Simulate a full CI/CD run
pipeline = CICDPipeline("abc123def456", "main")

pipeline.add_stage("Lint & Format", [
    {"name": "ruff check src/", "passes": True, "duration_sec": 3},
    {"name": "black --check src/", "passes": True, "duration_sec": 2},
    {"name": "mypy src/", "passes": True, "duration_sec": 8},
])

pipeline.add_stage("Unit Tests", [
    {"name": "pytest tests/unit/ (47 tests)", "passes": True, "duration_sec": 12},
    {"name": "Coverage check (>80%)", "passes": True, "duration_sec": 1},
])

pipeline.add_stage("Bundle Validation", [
    {"name": "databricks bundle validate --target dev", "passes": True, "duration_sec": 5},
    {"name": "databricks bundle validate --target prod", "passes": True, "duration_sec": 5},
])

pipeline.add_stage("Deploy to DEV", [
    {"name": "databricks bundle deploy --target dev", "passes": True, "duration_sec": 45},
    {"name": "pytest tests/integration/ --target=dev", "passes": True, "duration_sec": 120},
])

pipeline.add_stage("Deploy to PROD (manual approval)", [
    {"name": "Manual approval received", "passes": True, "duration_sec": 0},
    {"name": "databricks bundle deploy --target prod", "passes": True, "duration_sec": 45},
    {"name": "pytest tests/smoke/ --target=prod", "passes": True, "duration_sec": 30},
])

pipeline.run()


In [0]:
# ============================================
# 🎯 YOUR TURN: Design a CI/CD Pipeline
# ============================================
# Design a CI/CD pipeline for a data engineering team with:
# - 5 engineers, 3 environments (dev/staging/prod)
# - Python transformations + SQL models + DABs bundles
# - Must run tests before any deployment
# - Prod deployments require 2 approvals
# - Rollback must be possible within 5 minutes
#
# Design:
# 1. What CI stages run on every PR?
# 2. What CD stages run on merge to main?
# 3. How do you handle rollback?
# 4. What secrets/credentials are needed?
# 5. How do you test data quality in CI?


In [0]:
# ============================================
# ✅ SOLUTION
# ============================================
cicd_design = {
    "ci_stages_on_pr": [
        "1. Lint (ruff, black) -- fast feedback on code style",
        "2. Type check (mypy) -- catch type errors early",
        "3. Unit tests (pytest tests/unit/) -- test transformation logic",
        "4. Coverage check (>80% required) -- ensure tests are meaningful",
        "5. Bundle validate (dev + prod) -- catch config errors",
        "6. Security scan (bandit) -- catch hardcoded secrets",
    ],
    "cd_stages_on_merge": [
        "1. Deploy to DEV (automatic) -- always deploy to dev",
        "2. Integration tests on DEV -- test with real data",
        "3. Deploy to STAGING (automatic if DEV passes)",
        "4. Regression tests on STAGING -- full test suite",
        "5. Deploy to PROD (requires 2 approvals in GitHub)",
        "6. Smoke tests on PROD -- verify critical paths",
        "7. Create release tag -- for rollback reference",
    ],
    "rollback_strategy": {
        "method": "Git tag + DABs deploy",
        "steps": [
            "1. Identify last good release tag (e.g., release-20240315-060000)",
            "2. git checkout release-20240315-060000",
            "3. databricks bundle deploy --target prod",
            "4. Verify smoke tests pass",
            "5. Total time: < 5 minutes",
        ],
        "prevention": "Feature flags for risky changes (deploy disabled, enable when ready)"
    },
    "secrets_needed": {
        "DATABRICKS_HOST_DEV": "Dev workspace URL",
        "DATABRICKS_TOKEN_DEV": "Service principal token for dev",
        "DATABRICKS_HOST_STAGING": "Staging workspace URL",
        "DATABRICKS_TOKEN_STAGING": "Service principal token for staging",
        "DATABRICKS_HOST_PROD": "Prod workspace URL",
        "DATABRICKS_TOKEN_PROD": "Service principal token for prod",
        "SLACK_WEBHOOK": "For deployment notifications",
    },
    "data_quality_in_ci": [
        "Unit tests: test transformation logic with synthetic data",
        "Integration tests: run quality checks on DEV data sample",
        "Contract tests: validate schema against data contracts",
        "Regression tests: compare output metrics to baseline",
    ]
}

print("CI/CD PIPELINE DESIGN")
print("=" * 60)
for section, content in cicd_design.items():
    print(f"\n{section.upper().replace('_', ' ')}:")
    if isinstance(content, list):
        for item in content:
            print(f"  {item}")
    elif isinstance(content, dict):
        for key, value in content.items():
            if isinstance(value, list):
                print(f"  {key}:")
                for v in value:
                    print(f"    {v}")
            else:
                print(f"  {key}: {value}")


---
# 📋 Summary

## What You Learned

| Topic | Key Takeaway |
|---|---|
| **Git for DE** | Repository structure, branching, .gitignore |
| **Code Organization** | Modular pipelines > monolithic scripts |
| **Environments** | Same code + different config = safe deployments |
| **Testing** | Unit → Integration → Contract → E2E pyramid |
| **CI Pipeline** | Automated quality gates on every commit |
| **CD Pipeline** | Deploy → smoke test → promote (or rollback) |
| **Migrations** | Schema changes as versioned, ordered code |
| **Feature Flags** | Gradual rollout, kill switches, A/B testing |
| **DABs** | Declarative Automation Bundles for Databricks deployment |

## Key Principles
1. **Never deploy without tests** — automated quality gates prevent incidents
2. **Same code, different config** — environments differ only in configuration
3. **Small, frequent deploys** — easier to debug and rollback
4. **Everything in Git** — code, configs, schemas, tests, migrations
5. **Automate everything** — manual steps are error-prone steps

## Next Steps
- **Notebook 29:** dbt Concepts for Data Engineering
- **Notebook 35:** Deep dive into Declarative Automation Bundles (DABs)

---
*Notebook 28 of the Databricks Data Engineering series*
*Study Order: 18*